# 06 · Extended analysis for the PMC resubmission (WP-F)

**Project:** RecGym axis-reduction study (AxisFit) · **Notebook:** `06_analysis_extended.ipynb` · **Trains nothing.** It only reads result CSVs, so it needs no GPU and finishes in a few minutes on a Kaggle CPU session.

## What it does

It re-analyses the primary grid together with everything produced since: the extended lattice (notebook 04: A1 drop-one and A2 mixed pairs), the extra seeds (04b classical seeds 43/44, 02 CNN seeds 43-46) and the consistent-machine latency run (05).

| Block | Content | Handoff item |
|---|---|---|
| 0-2 | Input staging, loading, integrity checks, anchor and label-shuffle checks | acceptance criteria |
| 3 | Reproduction of the published primary numbers (guards against loading the wrong files) | WP-F |
| 4 | **E1 / E2** families: drop-one and mixed pairs vs 6-axis, Holm per position x model; drop-one cost table | WP-A |
| 5 | Best / median / worst per channel count over *all* configurations, plus a leave-one-subject-out selection-optimism estimate | WP-C1 |
| 6 | Mixed vs pure pairs: selection-free contrasts, the pre-declared `mix-AxGz` contrast, complementarity | WP-A |
| 7 | Marginal axis gain, original definition (reproduced) and the wider definition, reported per lattice transition | WP-A |
| 8 | Model-ranking agreement (Spearman, Kendall's W, bootstrap over subjects) | WP-F |
| 9 | Seed variability, underfit flags, seed-robustness of the primary claims, margin map | WP-B |
| 10 | Per-class effects of the extended configurations | RQ4 |
| 11 | Systems view: latency, size, analytic data volume, Pareto sets | WP-D |
| 12 | Extended master table, claims re-check, claims-to-sources manifest, findings digest | WP-F, 6.4 |

## Frozen design and analysis plan (declared before this notebook was run)

* **Protocol inherited unchanged:** LOSO over 10 subjects, 4 s windows, 50 % overlap, primary metric macro-F1 over 12 labels including `Null`, sensitivity metric macro-F1 over the 11 workouts. Nothing is retrained or re-tuned here.
* **Primary family is unchanged:** the 14 reduced primary configurations versus 6-axis, Holm within position x model. Adding configurations never changes those p-values.
* **New families (Holm per position x model):** E1 = the 6 drop-one configurations vs 6-axis (6 tests per cell); E2 = the 9 mixed pairs vs 6-axis (9 tests per cell). E3 (mixed triples) was **not run**, so it is not reported.
* **Statistics:** two-sided Wilcoxon signed-rank on the 10 paired subject-level values, percentile bootstrap 95 % CI over subjects (10,000 resamples, same resamples for every comparison), Cohen's dz, rank-biserial r, sign counts. Smallest attainable exact two-sided p at n = 10 is 0.00195. Descriptive non-inferiority margins 0.02, 0.03 and 0.05 (lower CI bound above minus margin).
* **Exploratory contrast (declared in the plan):** `mix-AxGz` vs the best pure pair per placement. The best pure pair is chosen on the same folds, which inflates the comparator, so an advantage of `mix-AxGz` is read conservatively. Reported with paired difference, CI and sign counts; Holm is applied only inside the labelled exploratory family.
* **Selection-bias control:** best, median and worst macro-F1 over all configurations of each channel count, so no conclusion rests on a best-of selection alone. A leave-one-subject-out *selection* estimate quantifies the optimism of best-of numbers without retraining. It is **approximate**: the stored per-subject scores of the other folds come from models that were trained on the held-out subject, so it is not a true nested LOSO (WP-C2 was not run).
* **Marginal gain, wider definition:** the lattice is partial (no mixed triples, no 4-channel subsets), so the wider definition is reported **per transition** (1->2, 2->3 pure only, 5->6) and is never blended across transitions.
* **No energy claim.** Channel count sets the data to sample, send, store and process. Latency is single-window, batch 1, one thread, one session, one fold per unit, and is **not** edge-device latency.

## Known limits that this notebook reports instead of hiding

* A3/A4 not run: the 3-channel level contains only the two pure triples, and no 4-channel configuration exists. Best/median/worst is complete only for k = 1, 2, 5, 6.
* The extended configurations have one seed (42). Seed variability is used as a noise reference (columns `seed_sd_ref`, `effect_gt_3sd`).
* The extension run's XGBoost 6-axis anchor differs slightly from the primary value (see the anchor table). XGBoost is therefore compared with a small hardware or run-to-run offset, which the seed SD helps to size.
* Axes are device-frame channels and are never read as anatomical directions.

## How to run on Kaggle

1. Add `AxisFit-main.zip` and `NewExtension.zip` as Kaggle datasets (or upload them to the notebook). Nested zips are unpacked automatically, so nothing has to be extracted by hand.
2. Run all cells. Optional environment variables: `INPUT_DIR`, `RESULTS_DIR`, `QUICK=1` (fewer bootstrap resamples, a smoke test of about a minute; the full run uses the frozen 10,000).
3. Download `recgym_06_analysis_extended.zip` from the last cell. Every number in the digest is computed, none is typed in.

In [ ]:
# =========================== PARAMETERS (edit here only) ===========================
import os
INPUT_DIR   = os.environ.get("INPUT_DIR", "")       # optional extra folder to search for the uploaded zips / folders
RESULTS_DIR = os.environ.get("RESULTS_DIR", "")     # optional output folder
QUICK       = int(os.environ.get("QUICK", "0"))     # 1 = smoke test with fewer bootstrap resamples

SEED = 42                                    # bootstrap seed, identical to notebook 03
N_BOOT = 1000 if QUICK else 10000            # frozen value is 10000
N_BOOT_RANK = 300 if QUICK else 2000         # bootstrap resamples for ranking-agreement intervals
ALPHA = 0.05
DELTAS = [0.02, 0.03, 0.05]                  # descriptive non-inferiority margins in macro-F1
MARGIN = 0.05                                # the margin used in the paper's claims
PRIMARY_METRIC = "macro_f1"                  # 12 labels incl. Null
SENS_METRIC = "macro_f1_workouts"            # 11 workouts, Null excluded
METRICS = [PRIMARY_METRIC, SENS_METRIC]
FS = 20                                      # nominal Hz
HOP_S = 40 / FS                              # window step in seconds (50 % overlap of 80 samples)
LOSS_FLAG = 0.5                              # CNN final train loss above this is flagged as underfit
EXPECTED_DATA_SHA = "2031c077a57e93d6b5c39da198470cb6c9b1a3c0652ab57cfc5d6a39bfa840e9"
DPI = 200
print("QUICK =", QUICK, "| N_BOOT =", N_BOOT)

In [ ]:
import sys, re, json, zipfile, shutil, hashlib, itertools, platform, warnings, math, tempfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd
import scipy, matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import wilcoxon, rankdata, spearmanr, chi2
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

ON_KAGGLE = Path("/kaggle/working").exists()
if RESULTS_DIR:
    OUT = Path(RESULTS_DIR)
elif ON_KAGGLE:
    OUT = Path("/kaggle/working/06_analysis_extended")
else:
    OUT = Path("results/analysis_extended")
WORK = Path(tempfile.gettempdir()) / "recgym_inputs"
for sub in ["tables", "figures", "checks"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

LOG = []
def log(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    LOG.append(s)

SAVED_TABLES = []
def save_table(t, name):
    p = OUT / "tables" / f"{name}.csv"
    t.to_csv(p, index=False)
    SAVED_TABLES.append(name)
    log("saved table:", p.name, t.shape)
    return t

def savefig(fig, name):
    fig.savefig(OUT / "figures" / f"{name}.png", dpi=DPI, bbox_inches="tight")
    fig.savefig(OUT / "figures" / f"{name}.pdf", bbox_inches="tight")
    log("saved figure:", name)
    plt.show()
    plt.close(fig)

CHECKS = []
def check(name, cond, detail="", hard=False):
    cond = bool(cond)
    CHECKS.append({"check": name, "passed": cond, "hard": hard, "detail": str(detail)})
    log(("PASS " if cond else ("FAIL " if hard else "WARN ")) + name + (f" | {detail}" if detail else ""))
    if hard:
        assert cond, f"hard integrity check failed: {name} | {detail}"

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

log("Versions: python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
    "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)
log("Output folder:", OUT.resolve())

In [ ]:
# ---------------------------- stage the inputs (unzip nested zips, index files) ----------------------------
def candidate_roots():
    roots = []
    for r in [INPUT_DIR, "/kaggle/input", "/content", "/mnt/user-data/uploads", "."]:
        if r and Path(r).exists():
            rp = Path(r).resolve()
            if rp not in roots:
                roots.append(rp)
    return roots

def skipped(p):
    s = p.resolve().as_posix()
    return ("__MACOSX" in s) or s.startswith(OUT.resolve().as_posix())

def extract_all_zips(max_rounds=3):
    extracted = {}
    for _ in range(max_rounds):
        new = 0
        for b in candidate_roots() + [WORK / "unz"]:
            if not b.exists():
                continue
            for z in b.rglob("*.zip"):
                zp = z.resolve()
                if skipped(zp) or zp in extracted:
                    continue
                if zp.stat().st_size > 3_000_000_000:
                    continue
                dest = WORK / "unz" / (hashlib.md5(zp.as_posix().encode()).hexdigest()[:8] + "_" + zp.stem)
                try:
                    with zipfile.ZipFile(zp) as zf:
                        zf.extractall(dest)
                    extracted[zp] = dest
                    new += 1
                except Exception as e:
                    log("could not extract", zp, "|", e)
                    extracted[zp] = None
        if new == 0:
            break
    return extracted

EXTRACTED = extract_all_zips()
log("Unpacked zips:", len([v for v in EXTRACTED.values() if v is not None]))

FILE_INDEX = {}
for b in candidate_roots() + [WORK / "unz"]:
    if not b.exists():
        continue
    for p in b.rglob("*"):
        if (not p.is_file()) or p.suffix.lower() not in (".csv", ".json", ".txt"):
            continue
        if skipped(p):
            continue
        FILE_INDEX.setdefault((p.parent.name, p.name), [])
        rp = p.resolve()
        if rp not in FILE_INDEX[(p.parent.name, p.name)]:
            FILE_INDEX[(p.parent.name, p.name)].append(rp)

INPUT_FILES = {}
def locate(parent, name, required=True):
    hits = sorted(FILE_INDEX.get((parent, name), []), key=lambda q: (len(q.as_posix()), q.as_posix()))
    if not hits:
        if required:
            have = sorted({k[0] for k in FILE_INDEX if k[1] == name})
            raise FileNotFoundError(f"{parent}/{name} not found. Folders that contain a {name}: {have}. "
                                    "Add AxisFit-main.zip and NewExtension.zip as inputs (or set INPUT_DIR).")
        return None
    if len(hits) > 1:
        log("note:", len(hits), "copies of", f"{parent}/{name}", "- using", hits[0])
    INPUT_FILES[f"{parent}/{name}"] = hits[0].as_posix()
    return hits[0]

# required result tables
P_CLS = locate("classical_full", "results.csv")
P_CNN = locate("cnn_full", "results.csv")
P_EXT = locate("extended_full", "results.csv")
P_C43 = locate("classical_seed43", "results.csv")
P_C44 = locate("classical_seed44", "results.csv")
P_CNNS = locate("cnn_seeds43-46_full", "results.csv")
P_LAT = locate("latency_full", "results.csv")
for k, v in INPUT_FILES.items():
    log("input:", k, "->", v)

In [ ]:
# ---------------------------- load and normalise ----------------------------
def norm(d, default_seed=None, default_group="P"):
    d = d.copy()
    for c in ["position", "model", "config", "test_subject"]:
        d[c] = d[c].astype(str)
    if "seed" not in d.columns:
        d["seed"] = default_seed
    d["seed"] = d["seed"].astype(int)
    if "group" not in d.columns:
        d["group"] = default_group
    return d

cls_p = norm(pd.read_csv(P_CLS), 42)
cnn_p = norm(pd.read_csv(P_CNN), 42)
ext   = norm(pd.read_csv(P_EXT), 42, "ext")
c43   = norm(pd.read_csv(P_C43), 43)
c44   = norm(pd.read_csv(P_C44), 44)
cnn_s = norm(pd.read_csv(P_CNNS), None)
lat   = norm(pd.read_csv(P_LAT), 42, "P")

IMU = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
POS = ["wrist", "leg", "pocket"]
MODELS = ["RF", "SVM", "XGB", "CNN"]
SUBJECTS = sorted(cls_p.test_subject.unique(), key=int)
F1COLS = [c for c in cls_p.columns if c.startswith("f1_")]
CLASSES = [c[3:] for c in F1COLS]

# ---- configuration lattice ----
def cfg_axes(name):
    if name == "6-axis": return frozenset(IMU)
    if name == "acc-3": return frozenset(["Ax", "Ay", "Az"])
    if name == "gyro-3": return frozenset(["Gx", "Gy", "Gz"])
    if name.startswith("drop-"): return frozenset(IMU) - {name[5:]}
    return frozenset(re.findall(r"[AG][xyz]", name))

def cfg_kind(name):
    ax = cfg_axes(name); n = len(ax)
    acc = sum(a[0] == "A" for a in ax)
    if n == 6: return "full"
    if n == 5: return "drop-one"
    if n == 1: return "single-acc" if acc == 1 else "single-gyro"
    if acc in (0, n):
        if n == 2: return "acc-pair" if acc == 2 else "gyro-pair"
        return "pure-triple"
    return "mixed-pair" if n == 2 else ("mixed-triple" if n == 3 else "mixed-4")

P15 = ["6-axis", "acc-3", "gyro-3", "acc-AxAy", "acc-AxAz", "acc-AyAz", "gyro-GxGy", "gyro-GxGz", "gyro-GyGz",
       "single-Ax", "single-Ay", "single-Az", "single-Gx", "single-Gy", "single-Gz"]
P_REDUCED = P15[1:]
A1 = [f"drop-{a}" for a in IMU]
A2 = [f"mix-A{i}G{j}" for i in "xyz" for j in "xyz"]
ALL30 = (["6-axis"] + A1 + ["acc-3", "gyro-3", "acc-AxAy", "acc-AxAz", "acc-AyAz", "gyro-GxGy", "gyro-GxGz", "gyro-GyGz"]
         + A2 + ["single-Ax", "single-Ay", "single-Az", "single-Gx", "single-Gy", "single-Gz"])
NCH = {c: len(cfg_axes(c)) for c in ALL30}
KIND = {c: cfg_kind(c) for c in ALL30}
AX2CFG = {cfg_axes(c): c for c in ALL30}
assert len(AX2CFG) == 30, "two configurations share an axis set"
ACC_PAIRS = [c for c in ALL30 if KIND[c] == "acc-pair"]
GYR_PAIRS = [c for c in ALL30 if KIND[c] == "gyro-pair"]
PURE_PAIRS = ACC_PAIRS + GYR_PAIRS
MIXED = [c for c in ALL30 if KIND[c] == "mixed-pair"]
assert (len(ACC_PAIRS), len(GYR_PAIRS), len(MIXED)) == (3, 3, 9)

# ---- unified seed-42 grid: primary (15 configs x 4 models) + extended (15 configs x 4 models) ----
KEEP = ["position", "model", "config", "group", "seed", "fold", "test_subject", "n_channels", "channels",
        "n_features", "macro_f1", "macro_f1_workouts", "accuracy", "weighted_f1"] + F1COLS
U = pd.concat([cls_p[KEEP], cnn_p[KEEP], ext[KEEP]], ignore_index=True)
assert set(U.seed.unique()) == {42}
assert set(U.config.unique()) == set(ALL30), set(U.config.unique()) ^ set(ALL30)
GROUP = U.drop_duplicates("config").set_index("config").group.to_dict()
log("Unified seed-42 grid:", U.shape, "| subjects:", SUBJECTS)

In [ ]:
# ---------------------------- integrity checks (hard = stops the notebook) ----------------------------
exp = {"classical primary": (cls_p, 3 * 15 * 10 * 3), "CNN primary (seed 42)": (cnn_p, 3 * 15 * 10),
       "extended (A1+A2, 4 models)": (ext, 3 * 15 * 10 * 4), "classical seed 43": (c43, 3 * 15 * 10 * 3),
       "classical seed 44": (c44, 3 * 15 * 10 * 3), "CNN seeds 43-46": (cnn_s, 3 * 15 * 10 * 4),
       "latency (30 configs x 3 pos x 4 models)": (lat, 3 * 30 * 4)}
for nm, (d, n) in exp.items():
    check(f"unit count: {nm}", len(d) == n, f"{len(d)} rows, expected {n}", hard=True)

KEYS = ["position", "model", "config", "seed", "test_subject"]
for nm, d in [("classical primary", cls_p), ("CNN primary", cnn_p), ("extended", ext), ("classical seed 43", c43),
              ("classical seed 44", c44), ("CNN seeds 43-46", cnn_s)]:
    check(f"no duplicate units: {nm}", d.duplicated(KEYS).sum() == 0, hard=True)
check("unified grid has no duplicate (position, model, config, subject)", U.duplicated(["position", "model", "config", "test_subject"]).sum() == 0, hard=True)
check("10 LOSO subjects in every file", all(sorted(d.test_subject.unique(), key=int) == SUBJECTS for d in [cls_p, cnn_p, ext, c43, c44, cnn_s]), hard=True)
check("every (position, model, config) cell has 10 subjects", (U.groupby(["position", "model", "config"]).test_subject.nunique() == 10).all(), hard=True)

# same folds and same windows as the primary grid
fold_map = cls_p.drop_duplicates(["fold", "test_subject"]).set_index("fold").test_subject.to_dict()
for nm, d in [("extended", ext), ("classical seed 43", c43), ("classical seed 44", c44), ("CNN seeds 43-46", cnn_s)]:
    ok = all(fold_map[int(f)] == s for f, s in zip(d.fold, d.test_subject))
    check(f"fold -> held-out subject mapping identical to primary: {nm}", ok, hard=True)
ref_w = cls_p.groupby(["position", "test_subject"]).agg(tr=("n_train_windows", "first"), te=("n_test_windows", "first"))
for nm, d in [("extended", ext), ("classical seed 43", c43), ("CNN seeds 43-46", cnn_s)]:
    w = d.groupby(["position", "test_subject"]).agg(tr_min=("n_train_windows", "min"), tr_max=("n_train_windows", "max"),
                                                    te_min=("n_test_windows", "min"), te_max=("n_test_windows", "max"))
    w = w.join(ref_w)
    ok = ((w.te_min == w.te) & (w.te_max == w.te)).all()
    check(f"test-window counts equal the primary grid: {nm}", ok, hard=True)
# the CNN train window count can differ only by design if the notebook subsampled; the classical SVM cap is a model detail

# channel string agrees with the configuration name; classical feature count is 15 per channel
pairs = U[["config", "channels"]].drop_duplicates()
check("channel strings agree with configuration names",
      all(frozenset(re.findall(r"[AG][xyz]", ch)) == cfg_axes(c) for c, ch in zip(pairs.config, pairs.channels)), hard=True)
cl_all = pd.concat([cls_p, ext[ext.model != "CNN"], c43, c44], ignore_index=True)
check("classical n_features = 15 x n_channels (no cross-channel features)", (cl_all.n_features == 15 * cl_all.n_channels).all(), hard=True)
check("n_channels column agrees with configuration name", all(NCH[c] == n for c, n in zip(U.config, U.n_channels)), hard=True)
check("macro-F1 values lie in [0, 1]", U[["macro_f1", "macro_f1_workouts"]].apply(lambda s: s.between(0, 1)).all().all(), hard=True)
check("group labels: P for primary, A1 for drop-one, A2 for mixed pairs",
      all(GROUP[c] == ("A1" if c in A1 else "A2" if c in A2 else "P") for c in ALL30), hard=True)

# data hash and the label-shuffle / anchor controls recorded by the runs
def find_key(obj, key):
    if isinstance(obj, dict):
        if key in obj: return obj[key]
        for v in obj.values():
            r = find_key(v, key)
            if r is not None: return r
    elif isinstance(obj, list):
        for v in obj:
            r = find_key(v, key)
            if r is not None: return r
    return None

RUN_FOLDERS = ["classical_full", "cnn_full", "extended_full", "classical_seed43", "classical_seed44", "cnn_seeds43-46_full", "latency_full"]
shas = {}
for fo in RUN_FOLDERS:
    for fn in ("environment.json", "run_config.json"):
        pth = locate(fo, fn, required=False)
        if pth is not None:
            val = find_key(json.load(open(pth)), "data_sha256")
            if val:
                shas[fo] = val
                break
check("RecGym.csv SHA-256 recorded in the runs equals the frozen hash (where recorded)",
      len(shas) > 0 and all(v == EXPECTED_DATA_SHA for v in shas.values()), f"{len(shas)} of {len(RUN_FOLDERS)} runs record it: {sorted(shas)}")

anch = []
for fo in ["extended_full", "classical_seed43", "classical_seed44"]:
    pth = locate(fo, "anchor_check.csv", required=False)
    if pth is not None:
        a = pd.read_csv(pth); a.insert(0, "run", fo); anch.append(a)
ANCHOR = pd.concat(anch, ignore_index=True) if anch else pd.DataFrame()
if len(ANCHOR):
    save_table(ANCHOR, "anchor_checks")
    display(ANCHOR.round(5))
    for _, r in ANCHOR[~ANCHOR.within_tolerance.astype(bool)].iterrows():
        check(f"anchor within tolerance: {r['run']} {r['model']} {r['position']} fold {r['fold']} {r['config']}", False,
              f"|diff| = {r['abs_diff']:.4f} (tolerance {r['tolerance']}); non-blocking, see seed SD in block 9")

shuf = []
for fo in ["cnn_full", "extended_full", "classical_seed43", "classical_seed44", "cnn_seeds43-46_full"]:
    pth = locate(fo, "sanity_checks.json", required=False)
    if pth is None:
        continue
    js = json.load(open(pth))
    for mdl, v in js.items():
        if isinstance(v, dict) and "label_shuffle_macro_f1" in v:
            shuf.append(dict(run=fo, model=mdl, label_shuffle_macro_f1=v["label_shuffle_macro_f1"],
                             chance=v.get("chance_reference", np.nan), threshold=v.get("threshold", np.nan)))
SHUF = pd.DataFrame(shuf)
if len(SHUF):
    save_table(SHUF, "label_shuffle_controls")
    display(SHUF.round(4))
    check("label-shuffle controls stay below the run thresholds", (SHUF.label_shuffle_macro_f1 < SHUF.threshold.fillna(0.25)).all())

In [ ]:
# ---------------------------- statistics helpers (same logic as notebook 03) ----------------------------
MIN_P = 2 / 2 ** 10
log(f"Smallest possible exact two-sided Wilcoxon p with n = 10 pairs: {MIN_P:.5f}")
_BOOT = {}
def boot_idx(n):
    if n not in _BOOT:
        _BOOT[n] = np.random.default_rng(SEED).integers(0, n, size=(N_BOOT, n))
    return _BOOT[n]

def paired_stats(d):
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    n = len(d)
    sd = d.std(ddof=1) if n > 1 else np.nan
    nz = d[d != 0]
    if n < 2:
        return dict(n=n, mean_diff=np.nan, median_diff=np.nan, ci_lo=np.nan, ci_hi=np.nan, dz=np.nan, rbc=np.nan,
                    n_pos=int((d > 0).sum()), n_neg=int((d < 0).sum()), n_zero=int((d == 0).sum()), p=np.nan)
    if len(nz) == 0:
        p, rbc = 1.0, 0.0
    else:
        r = rankdata(np.abs(nz))
        wp, wm = r[nz > 0].sum(), r[nz < 0].sum()
        rbc = (wp - wm) / (wp + wm)
        try:
            p = float(wilcoxon(d, zero_method="wilcox", alternative="two-sided", method="auto").pvalue)
        except ValueError:
            p = 1.0
    bm = d[boot_idx(n)].mean(axis=1)
    lo, hi = np.percentile(bm, [2.5, 97.5])
    return dict(n=n, mean_diff=float(d.mean()), median_diff=float(np.median(d)), ci_lo=float(lo), ci_hi=float(hi),
                dz=float(d.mean() / sd) if sd and sd > 0 else np.nan, rbc=float(rbc),
                n_pos=int((d > 0).sum()), n_neg=int((d < 0).sum()), n_zero=int((d == 0).sum()), p=p)

def holm(p):
    p = np.asarray(p, float)
    out = np.full(len(p), np.nan)
    ok = np.isfinite(p)
    pv = p[ok]
    m = len(pv)
    order = np.argsort(pv)
    adj = np.empty(m)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * pv[i])
        adj[i] = min(1.0, run)
    out[ok] = adj
    return out

def add_holm(t, fam_col):
    t = t.copy().reset_index(drop=True)
    t["p_holm_family"] = np.nan
    for _, idx in t.groupby(fam_col).groups.items():
        t.loc[idx, "p_holm_family"] = holm(t.loc[idx, "p"].values)
    t["sig_family"] = t["p_holm_family"] < ALPHA
    return t

def add_ni(t):
    t = t.copy()
    for dl in DELTAS:
        t["ni_" + str(dl).replace(".", "p")] = t.ci_lo > -dl
    return t

def build_V(metric, frame=None):
    f = U if frame is None else frame
    V = f.pivot_table(index=["position", "model", "config"], columns="test_subject", values=metric, aggfunc="mean")
    return V.reindex(columns=SUBJECTS)

def vec(V, pos, model, cfg):
    return V.loc[(pos, model, cfg)].values.astype(float)

VM = {m: build_V(m) for m in METRICS}
for m in METRICS:
    check(f"subject x unit matrix complete ({m})", VM[m].shape == (3 * 4 * 30, 10) and not VM[m].isna().any().any(), VM[m].shape, hard=True)
V1 = VM[PRIMARY_METRIC]

def reduction_table(Vm, cfgs, family, metric, models=None):
    models = models or MODELS
    rows = []
    for pos in POS:
        for m in models:
            ref = vec(Vm, pos, m, "6-axis")
            for c in cfgs:
                x = vec(Vm, pos, m, c)
                rows.append(dict(metric=metric, family=family, position=pos, model=m, config=c, kind=KIND[c],
                                 n_channels=NCH[c], data_rate_hz=NCH[c] * FS, mean_6axis=ref.mean(), mean_cfg=x.mean(),
                                 sd_cfg=x.std(ddof=1), retained_pct=100 * x.mean() / ref.mean(), **paired_stats(x - ref)))
    T = pd.DataFrame(rows)
    T["fam"] = T.position + "|" + T.model
    T = add_holm(T, "fam")
    T = add_ni(T)
    T["sig_decrease"] = (T.p_holm_family < ALPHA) & (T.mean_diff < 0)
    return T
log("helpers ready")

## Block 3 · Reproduce the published primary numbers

Before anything new is computed, the primary family (14 reduced configurations vs 6-axis, Holm within position x model) is recomputed from the raw per-subject results and compared with `master_results.csv` from the repository. If these do not match, the wrong files were loaded.

In [ ]:
CLAIMS = []
def claim(cid, text, value, source):
    CLAIMS.append(dict(claim_id=cid, statement=text, value=str(value), source_table=source))

A_PRIM = pd.concat([reduction_table(VM[m], P_REDUCED, "primary14", m) for m in METRICS], ignore_index=True)
save_table(A_PRIM, "A_primary_reduction_vs_6axis")
Ap = A_PRIM[A_PRIM.metric == PRIMARY_METRIC]
n_sig_primary = int(Ap.sig_decrease.sum())
log(f"Primary grid (seed 42): {n_sig_primary} of {len(Ap)} reduced-vs-6-axis comparisons are Holm-significant decreases")
check("primary grid has 168 comparisons (14 configs x 3 positions x 4 models)", len(Ap) == 168, len(Ap), hard=True)

mp = locate("results", "master_results.csv", required=False)
if mp is not None:
    M0 = pd.read_csv(mp)
    M0 = M0.rename(columns={"placement": "position", "axis_config": "config"})
    M0["position"] = M0.position.astype(str); M0["model"] = M0.model.astype(str); M0["config"] = M0.config.astype(str)
    cmp_ = Ap.merge(M0[["position", "model", "config", "macro_f1_mean", "macro_f1_diff_vs_6axis", "diff_ci_lo", "diff_ci_hi", "p_holm_family"]],
                    on=["position", "model", "config"], how="left", suffixes=("", "_master"))
    check("every primary cell found in master_results.csv", cmp_.macro_f1_mean.notna().all(), hard=True)
    d_mean = (cmp_.mean_cfg - cmp_.macro_f1_mean).abs().max()
    d_diff = (cmp_.mean_diff - cmp_.macro_f1_diff_vs_6axis).abs().max()
    d_p = (cmp_.p_holm_family - cmp_.p_holm_family_master).abs().max()
    d_ci = max((cmp_.ci_lo - cmp_.diff_ci_lo).abs().max(), (cmp_.ci_hi - cmp_.diff_ci_hi).abs().max())
    check("reproduces master_results.csv: cell means", d_mean < 5e-6, f"max |diff| = {d_mean:.2e}")
    check("reproduces master_results.csv: paired differences vs 6-axis", d_diff < 5e-6, f"max |diff| = {d_diff:.2e}")
    check("reproduces master_results.csv: Holm-adjusted p", d_p < 5e-5, f"max |diff| = {d_p:.2e}")
    check("reproduces master_results.csv: bootstrap CI bounds", d_ci < 5e-4, f"max |diff| = {d_ci:.2e} (needs N_BOOT = 10000 to match)" if QUICK else f"max |diff| = {d_ci:.2e}")
    n_master = int(((M0.p_holm_family < ALPHA) & (M0.macro_f1_diff_vs_6axis < 0)).sum())
    check("Holm-significant decrease count equals master_results.csv", n_master == n_sig_primary, f"recomputed {n_sig_primary}, master {n_master}")
else:
    log("master_results.csv not found: reproduction check against the repository skipped")
claim("primary_n_sig", "Primary grid: Holm-significant decreases among 168 reduced-vs-6-axis comparisons (seed 42)", n_sig_primary, "A_primary_reduction_vs_6axis")

## Block 4 · Extended lattice: E1 (drop-one) and E2 (mixed pairs) vs 6-axis

Holm is applied separately per position x model (6 tests in E1, 9 in E2), exactly as frozen. The 6-axis reference is the primary-grid 6-axis result, paired by subject. The primary family above is untouched.

In [ ]:
E1 = pd.concat([reduction_table(VM[m], A1, "E1_drop_one", m) for m in METRICS], ignore_index=True)
E2 = pd.concat([reduction_table(VM[m], A2, "E2_mixed_pairs", m) for m in METRICS], ignore_index=True)
EE = pd.concat([E1, E2], ignore_index=True)

# seed-noise reference for the extension (one seed only there): median seed SD per model is computed in block 9
save_table(E1, "E1_drop_one_vs_6axis")
save_table(E2, "E2_mixed_pairs_vs_6axis")

Ep = EE[EE.metric == PRIMARY_METRIC]
summ = (Ep.groupby(["family", "position", "model"])
          .agg(n_tests=("p", "size"), n_sig_decrease=("sig_decrease", "sum"), n_meet_margin_0p05=("ni_0p05", "sum"),
               best_cfg_mean_diff=("mean_diff", "max"), worst_cfg_mean_diff=("mean_diff", "min")).reset_index())
save_table(summ, "E_summary_by_cell")
print("E1 and E2 summary by position x model (primary metric):")
display(summ.round(3))

tot = Ep.groupby("family").agg(n_tests=("p", "size"), n_sig_decrease=("sig_decrease", "sum"), n_meet_margin_0p05=("ni_0p05", "sum"))
display(tot)
claim("E1_n_sig", "E1 drop-one: Holm-significant decreases among 72 tests", int(Ep[Ep.family == "E1_drop_one"].sig_decrease.sum()), "E1_drop_one_vs_6axis")
claim("E2_n_sig", "E2 mixed pairs: Holm-significant decreases among 108 tests", int(Ep[Ep.family == "E2_mixed_pairs"].sig_decrease.sum()), "E2_mixed_pairs_vs_6axis")
claim("E1_n_margin", "E1 drop-one: tests whose CI lower bound exceeds -0.05", int(Ep[Ep.family == "E1_drop_one"].ni_0p05.sum()), "E1_drop_one_vs_6axis")
claim("E2_n_margin", "E2 mixed pairs: tests whose CI lower bound exceeds -0.05", int(Ep[Ep.family == "E2_mixed_pairs"].ni_0p05.sum()), "E2_mixed_pairs_vs_6axis")

# Null-excluded sensitivity concordance for the extension
a = EE[EE.metric == PRIMARY_METRIC].set_index(["family", "position", "model", "config"])
b = EE[EE.metric == SENS_METRIC].set_index(["family", "position", "model", "config"])
j = a[["mean_diff", "sig_decrease"]].join(b[["mean_diff", "sig_decrease"]], rsuffix="_sens")
conc = []
for fam, g in j.groupby(level=0):
    conc.append(dict(family=fam, n=len(g), spearman_mean_diff=float(spearmanr(g.mean_diff, g.mean_diff_sens)[0]),
                     n_sig_primary=int(g.sig_decrease.sum()), n_sig_null_excluded=int(g.sig_decrease_sens.sum()),
                     n_discordant=int((g.sig_decrease != g.sig_decrease_sens).sum())))
CONC = save_table(pd.DataFrame(conc), "E_null_concordance")
display(CONC.round(4))

# drop-one cost: which removed axis hurts most, per position and model
dr = E1[E1.metric == PRIMARY_METRIC].copy()
dr["axis_removed"] = dr.config.str[5:]
DROP = dr[["position", "model", "axis_removed", "mean_diff", "ci_lo", "ci_hi", "retained_pct", "n_pos", "n_neg", "p_holm_family", "sig_decrease", "ni_0p05"]].copy()
DROP["rank_most_critical"] = DROP.groupby(["position", "model"]).mean_diff.rank(ascending=True, method="min")
save_table(DROP, "E1_drop_one_cost_by_axis")
worst_axis = (DROP[DROP.rank_most_critical == 1].groupby("position").axis_removed
              .agg(lambda s: s.value_counts().index[0]).to_dict())
agree = (DROP[DROP.rank_most_critical == 1].groupby("position").axis_removed.agg(lambda s: int(s.value_counts().iloc[0])).to_dict())
for pos in POS:
    log(f"{pos}: most damaging single axis to remove (modal over models) = {worst_axis[pos]} ({agree[pos]} of 4 models agree)")
    claim(f"drop_worst_{pos}", f"Most damaging axis to remove at {pos} (modal over models; models agreeing)", f"{worst_axis[pos]} ({agree[pos]}/4)", "E1_drop_one_cost_by_axis")
pivot_drop = DROP.pivot_table(index=["position", "axis_removed"], columns="model", values="mean_diff")
display(pivot_drop[MODELS].round(3))

## Block 5 · Selection-bias control: best, median and worst over all configurations of each size

Single best-of numbers are optimistic. Here every conclusion is also shown as the **median and worst** configuration of the same channel count, and the optimism of best-of is estimated by choosing the configuration **without the held-out subject** (approximate leave-one-subject-out selection on stored scores; see the plan for why it is not a true nested LOSO).

In [ ]:
from math import comb
rows = []
for pos in POS:
    for m in MODELS:
        ref_v = vec(V1, pos, m, "6-axis")
        for k in [6, 5, 3, 2, 1]:
            cs = [c for c in ALL30 if NCH[c] == k]
            means = pd.Series({c: vec(V1, pos, m, c).mean() for c in cs})
            rows.append(dict(position=pos, model=m, n_channels=k, n_configs=len(cs), n_possible=comb(6, k),
                             lattice_complete=(len(cs) == comb(6, k)),
                             best_cfg=means.idxmax(), best=means.max(), median=means.median(),
                             worst_cfg=means.idxmin(), worst=means.min(), mean_all=means.mean(),
                             mean_6axis=ref_v.mean(), best_retained_pct=100 * means.max() / ref_v.mean(),
                             median_retained_pct=100 * means.median() / ref_v.mean(), worst_retained_pct=100 * means.min() / ref_v.mean()))
BMW = save_table(pd.DataFrame(rows), "BMW_best_median_worst_by_channel_count")
cov = BMW.drop_duplicates("n_channels")[["n_channels", "n_configs", "n_possible", "lattice_complete"]]
print("Lattice coverage (configurations evaluated vs possible subsets of the six channels):")
display(cov)
display(BMW[BMW.model == "XGB"][["position", "n_channels", "n_configs", "best_cfg", "best", "median", "worst_cfg", "worst"]].round(3))

# leave-one-subject-out selection of the best configuration per channel count
rows = []
for pos in POS:
    for m in MODELS:
        ref_v = vec(V1, pos, m, "6-axis")
        for k in [5, 3, 2, 1]:
            cs = [c for c in ALL30 if NCH[c] == k]
            X = np.array([vec(V1, pos, m, c) for c in cs])
            ib = int(X.mean(axis=1).argmax())
            naive = X[ib].mean()
            sel, chosen = [], []
            for s in range(X.shape[1]):
                mask = np.arange(X.shape[1]) != s
                jb = int(X[:, mask].mean(axis=1).argmax())
                sel.append(X[jb, s]); chosen.append(cs[jb])
            sel = np.array(sel)
            st = paired_stats(sel - ref_v)
            rows.append(dict(position=pos, model=m, n_channels=k, n_configs=len(cs), naive_best_cfg=cs[ib], naive_best=naive,
                             loso_selected_mean=sel.mean(), optimism=naive - sel.mean(),
                             share_folds_same_choice=float(np.mean([c == cs[ib] for c in chosen])),
                             mean_all_configs=X.mean(), mean_6axis=ref_v.mean(),
                             loso_selected_diff_vs_6axis=st["mean_diff"], loso_ci_lo=st["ci_lo"], loso_ci_hi=st["ci_hi"],
                             loso_meets_margin_0p05=st["ci_lo"] > -MARGIN, n_subjects_better_than_6axis=st["n_pos"]))
SEL = save_table(pd.DataFrame(rows), "SEL_loso_selection_optimism")
print("Optimism of best-of numbers (naive best minus leave-one-subject-out selected), mean over positions and models:")
display(SEL.groupby("n_channels")[["optimism", "share_folds_same_choice"]].agg(["mean", "max"]).round(4))
claim("sel_optimism_max", "Largest best-of optimism (naive best minus LOSO-selected macro-F1) over all cells", f"{SEL.optimism.max():.4f}", "SEL_loso_selection_optimism")
claim("sel_optimism_median", "Median best-of optimism over all cells", f"{SEL.optimism.median():.4f}", "SEL_loso_selection_optimism")

## Block 6 · Mixed versus pure pairs

Three layers, from least to most selection-dependent:

1. **Selection-free:** mean over all 9 mixed pairs minus mean over the pure pairs (all 6, or the 3 accelerometer pairs, or the 3 gyroscope pairs). No best-of choice is involved. Holm family: model x contrast, three tests (positions).
2. **Best-of:** best mixed pair vs best pure pair, both chosen on the same folds (optimistic; shown for completeness).
3. **Declared exploratory contrast:** `mix-AxGz` vs the best pure pair and vs the mean of all pure pairs. The best pure pair is selected on the same folds, which favours the comparator, so any advantage of `mix-AxGz` is conservative. Holm family: model x contrast.

A separate **complementarity** view asks whether a pair beats its better single axis: `F1(pair) - max(F1(single a), F1(single b))`, compared between mixed and pure pairs.

In [ ]:
def grp_mean(V, pos, m, cs):
    return np.mean([vec(V, pos, m, c) for c in cs], axis=0)

rows = []
for pos in POS:
    for m in MODELS:
        for nm, (g1, g2) in {"mixed minus all pure pairs": (MIXED, PURE_PAIRS), "mixed minus accelerometer pairs": (MIXED, ACC_PAIRS),
                             "mixed minus gyroscope pairs": (MIXED, GYR_PAIRS)}.items():
            rows.append(dict(layer="1 selection-free", position=pos, model=m, contrast=nm, comparator="", **paired_stats(grp_mean(V1, pos, m, g1) - grp_mean(V1, pos, m, g2))))
        bm = max(MIXED, key=lambda c: vec(V1, pos, m, c).mean())
        bp = max(PURE_PAIRS, key=lambda c: vec(V1, pos, m, c).mean())
        rows.append(dict(layer="2 best-of (optimistic)", position=pos, model=m, contrast="best mixed pair minus best pure pair",
                         comparator=f"{bm} vs {bp}", **paired_stats(vec(V1, pos, m, bm) - vec(V1, pos, m, bp))))
        ax = vec(V1, pos, m, "mix-AxGz")
        rows.append(dict(layer="3 exploratory (declared)", position=pos, model=m, contrast="mix-AxGz minus best pure pair",
                         comparator=f"best pure pair on same folds = {bp}", **paired_stats(ax - vec(V1, pos, m, bp))))
        rows.append(dict(layer="3 exploratory (declared)", position=pos, model=m, contrast="mix-AxGz minus mean of all pure pairs",
                         comparator="mean of 6 pure pairs", **paired_stats(ax - grp_mean(V1, pos, m, PURE_PAIRS))))
MP = pd.DataFrame(rows)
MP["fam"] = MP.model + "|" + MP.contrast
MP = add_holm(MP, "fam")
save_table(MP, "MP_mixed_vs_pure_pairs")
show = MP[["layer", "position", "model", "contrast", "mean_diff", "ci_lo", "ci_hi", "n_pos", "n_neg", "p_holm_family"]]
print("Selection-free: mixed minus all pure pairs")
display(show[show.contrast == "mixed minus all pure pairs"].round(3))
print("Declared exploratory: mix-AxGz minus best pure pair (comparator inflated by same-fold selection)")
display(show[show.contrast == "mix-AxGz minus best pure pair"].round(3))

sf = MP[MP.contrast == "mixed minus all pure pairs"]
claim("mixed_selfree_pos_cells", "Cells (of 12) where mixed pairs beat the mean pure pair (mean difference > 0)", int((sf.mean_diff > 0).sum()), "MP_mixed_vs_pure_pairs")
claim("mixed_selfree_sig_cells", "Cells (of 12) where that difference is Holm-significant (either direction)", int((sf.p_holm_family < ALPHA).sum()), "MP_mixed_vs_pure_pairs")
ex = MP[MP.contrast == "mix-AxGz minus best pure pair"]
claim("axgz_ci_above0_cells", "Cells (of 12) where mix-AxGz minus best pure pair has a CI lower bound above 0", int((ex.ci_lo > 0).sum()), "MP_mixed_vs_pure_pairs")
claim("axgz_mean_diff_range", "mix-AxGz minus best pure pair: range of mean differences over 12 cells", f"{ex.mean_diff.min():.3f} to {ex.mean_diff.max():.3f}", "MP_mixed_vs_pure_pairs")

# best 2-channel configuration is mixed in how many cells
b2 = BMW[BMW.n_channels == 2]
n_mixed_best = int(b2.best_cfg.str.startswith("mix-").sum())
claim("best2_is_mixed_cells", "Cells (of 12) where the best 2-channel configuration is a mixed pair", n_mixed_best, "BMW_best_median_worst_by_channel_count")
log(f"Best 2-channel configuration is a mixed pair in {n_mixed_best} of 12 position x model cells")

# complementarity
def comp_vec(V, pos, m, c):
    singles = [vec(V, pos, m, f"single-{x}") for x in sorted(cfg_axes(c))]
    return vec(V, pos, m, c) - np.maximum.reduce(singles)

rows, prow = [], []
for pos in POS:
    for m in MODELS:
        cm = {c: comp_vec(V1, pos, m, c) for c in MIXED + PURE_PAIRS}
        for c, v in cm.items():
            prow.append(dict(position=pos, model=m, config=c, kind=KIND[c], mean_gain_over_better_single=v.mean()))
        d = np.mean([cm[c] for c in MIXED], axis=0) - np.mean([cm[c] for c in PURE_PAIRS], axis=0)
        rows.append(dict(position=pos, model=m, contrast="complementarity: mixed minus pure pairs", **paired_stats(d)))
COMP = pd.DataFrame(rows); COMP["fam"] = COMP.model
COMP = add_holm(COMP, "fam")
save_table(COMP, "MP_complementarity_mixed_vs_pure")
save_table(pd.DataFrame(prow), "MP_complementarity_by_pair")
display(COMP[["position", "model", "mean_diff", "ci_lo", "ci_hi", "n_pos", "n_neg", "p_holm_family"]].round(3))

## Block 7 · Marginal gain of an axis

**Definition 1 (original, reproduced):** adding an axis to a non-empty subset of the *same* sensor, averaged over the available subsets. Holm family per model (3 positions x 6 axes = 18 tests).
**Definition 2 (wider):** adding an axis to *any* non-empty subset for which both configurations exist. Because the lattice is partial (no mixed triples, no 4-channel subsets), only three transitions exist: 1->2 (all five partners, mixed or not), 2->3 (pure triples only) and 5->6 (adding the removed axis back to a drop-one configuration). Each transition is its own family and is **not** blended with the others.

In [ ]:
GROUPS = {"acc": ["Ax", "Ay", "Az"], "gyro": ["Gx", "Gy", "Gz"]}
rows = []
for pos in POS:
    for m in MODELS:
        for grp, axes in GROUPS.items():
            for a_ in axes:
                others = [b_ for b_ in axes if b_ != a_]
                gains = []
                for r in (1, 2):
                    for S in itertools.combinations(others, r):
                        gains.append(vec(V1, pos, m, AX2CFG[frozenset(S) | {a_}]) - vec(V1, pos, m, AX2CFG[frozenset(S)]))
                rows.append(dict(definition="1 same-sensor", transition="1->2 and 2->3 within sensor", position=pos, model=m, modality=grp,
                                 axis=a_, n_subsets=len(gains), **paired_stats(np.mean(gains, axis=0))))
for pos in POS:
    for m in MODELS:
        for a_ in IMU:
            rest = [b_ for b_ in IMU if b_ != a_]
            by_tr = {}
            for r in range(1, 6):
                for S in itertools.combinations(rest, r):
                    S = frozenset(S); T_ = S | {a_}
                    if S in AX2CFG and T_ in AX2CFG:
                        by_tr.setdefault((r, r + 1), []).append(vec(V1, pos, m, AX2CFG[T_]) - vec(V1, pos, m, AX2CFG[S]))
            for (r, r1), lst in sorted(by_tr.items()):
                rows.append(dict(definition="2 any subset", transition=f"{r}->{r1}", position=pos, model=m,
                                 modality="acc" if a_[0] == "A" else "gyro", axis=a_, n_subsets=len(lst),
                                 **paired_stats(np.mean(lst, axis=0))))
MG = pd.DataFrame(rows)
MG["fam"] = MG.definition + "|" + MG.transition + "|" + MG.model
MG = add_holm(MG, "fam")
save_table(MG, "MG_marginal_axis_gain")

# reproduction of the repository's definition-1 table (Table06), where available
t6 = locate("tables", "Table06_axis_marginal_gain.csv", required=False)
d1 = MG[MG.definition == "1 same-sensor"]
if t6 is not None:
    T6 = pd.read_csv(t6)
    ok = True; worst = 0.0
    for _, r in T6.iterrows():
        for a_ in IMU:
            txt = str(r[a_]).split(" ")[0]
            val = float(txt)
            mine = d1[(d1.position == r["position"]) & (d1.model == r["model"]) & (d1.axis == a_)].mean_diff.iloc[0]
            worst = max(worst, abs(mine - val))
    check("definition-1 marginal gains reproduce the repository's Table06 (3-decimal rounding)", worst < 0.0006, f"max |diff| = {worst:.4f}")

for tr in ["1->2", "2->3", "5->6"]:
    g = MG[(MG.definition == "2 any subset") & (MG.transition == tr)]
    top = (g.groupby(["position", "axis"]).mean_diff.mean().reset_index().sort_values("mean_diff", ascending=False)
             .groupby("position").head(2))
    print(f"Definition 2, transition {tr}: two largest mean gains per position (mean over models)")
    display(top.round(3))
top1 = (d1.groupby(["position", "axis"]).mean_diff.mean().reset_index().sort_values("mean_diff", ascending=False).groupby("position").head(2))
for pos in POS:
    t_ = top1[top1.position == pos]
    claim(f"mg1_top_{pos}", f"Definition-1 marginal gain, two largest axes at {pos} (mean over models)",
          "; ".join(f"{r.axis} {r.mean_diff:+.3f}" for r in t_.itertuples()), "MG_marginal_axis_gain")

## Block 8 · Do the four models agree on which configurations are good?

Per position, the configurations are ranked by their mean macro-F1 for each model. Agreement is summarised by the mean pairwise Spearman correlation and Kendall's W (tie-corrected), with a percentile bootstrap interval over subjects. Ranking over all 30 configurations is dominated by channel count, so the informative views are **within one channel count** (k = 2 with 15 configurations, k = 1 with 6, k = 5 with 6).

In [ ]:
def kendall_w(M):
    m, n = M.shape
    R = np.apply_along_axis(rankdata, 1, M)
    Rj = R.sum(axis=0)
    S = ((Rj - Rj.mean()) ** 2).sum()
    T = 0.0
    for i in range(m):
        _, cnt = np.unique(M[i], return_counts=True)
        T += (cnt ** 3 - cnt).sum()
    den = m ** 2 * (n ** 3 - n) - m * T
    return float(12 * S / den) if den > 0 else np.nan

def rho_matrix(M):
    return np.corrcoef(np.apply_along_axis(rankdata, 1, M))

def mean_offdiag(C):
    return float(C[np.triu_indices_from(C, k=1)].mean())

RANK_SETS = {"all 30 configs": ALL30, "15 primary configs": P15,
             "k=5 (6 drop-one)": [c for c in ALL30 if NCH[c] == 5],
             "k=2 (15 pairs)": [c for c in ALL30 if NCH[c] == 2],
             "k=1 (6 singles)": [c for c in ALL30 if NCH[c] == 1]}
rng = np.random.default_rng(SEED)
rows, prow, RHO = [], [], {}
for pos in POS:
    for sname, cs in RANK_SETS.items():
        X = np.array([[vec(V1, pos, m, c) for c in cs] for m in MODELS])      # models x configs x subjects
        M = X.mean(axis=2)
        W = kendall_w(M); C = rho_matrix(M); RHO[(sname, pos)] = C
        n = len(cs); mm = len(MODELS)
        bW, bR = [], []
        for _ in range(N_BOOT_RANK):
            idx = rng.integers(0, X.shape[2], size=X.shape[2])
            Mb = X[:, :, idx].mean(axis=2)
            bW.append(kendall_w(Mb)); bR.append(mean_offdiag(rho_matrix(Mb)))
        rows.append(dict(position=pos, config_set=sname, n_configs=n, kendall_w=W,
                         w_ci_lo=float(np.nanpercentile(bW, 2.5)), w_ci_hi=float(np.nanpercentile(bW, 97.5)),
                         chi2_p_approx=float(chi2.sf(mm * (n - 1) * W, n - 1)),
                         mean_pairwise_spearman=mean_offdiag(C), rho_ci_lo=float(np.nanpercentile(bR, 2.5)), rho_ci_hi=float(np.nanpercentile(bR, 97.5)),
                         min_pairwise_spearman=float(C[np.triu_indices_from(C, k=1)].min())))
        for i, j in itertools.combinations(range(mm), 2):
            prow.append(dict(position=pos, config_set=sname, model_a=MODELS[i], model_b=MODELS[j], spearman=float(C[i, j])))
RANK = save_table(pd.DataFrame(rows), "RANK_model_agreement")
save_table(pd.DataFrame(prow), "RANK_pairwise_spearman")
display(RANK.round(3))

rows = []
for pos in POS:
    for k in (5, 2, 1):
        cs = [c for c in ALL30 if NCH[c] == k]
        for m in MODELS:
            s = pd.Series({c: vec(V1, pos, m, c).mean() for c in cs}).sort_values(ascending=False)
            rows.append(dict(position=pos, n_channels=k, model=m, best=s.index[0], second=s.index[1], best_f1=s.iloc[0]))
TOPC = save_table(pd.DataFrame(rows), "RANK_top_configuration_by_model")
agreeing = (TOPC.groupby(["position", "n_channels"]).best.agg(lambda s: f"{s.value_counts().index[0]} ({int(s.value_counts().iloc[0])}/4 models)").reset_index())
display(agreeing)
for r in RANK[RANK.config_set == "k=2 (15 pairs)"].itertuples():
    claim(f"rank_w_k2_{r.position}", f"Kendall's W across 4 models over the 15 two-channel configurations at {r.position}",
          f"{r.kendall_w:.3f} (95% CI {r.w_ci_lo:.3f} to {r.w_ci_hi:.3f})", "RANK_model_agreement")

## Block 9 · Seed variability and robustness of the primary claims

CNN: seeds 42 (primary) plus 43-46. Classical models: seeds 42 (primary) plus 43 and 44. The extended configurations exist for seed 42 only, so the seed SD of the primary configurations is used as a **noise reference** for them. Underfit flags use the final training loss (the primary grid is not re-run with more epochs).

In [ ]:
CNN_ALL = pd.concat([cnn_p, cnn_s], ignore_index=True)
CLS_ALL = pd.concat([cls_p, c43, c44], ignore_index=True)
check("CNN seeds are 42-46", sorted(CNN_ALL.seed.unique()) == [42, 43, 44, 45, 46], sorted(CNN_ALL.seed.unique()), hard=True)
check("classical seeds are 42-44", sorted(CLS_ALL.seed.unique()) == [42, 43, 44], sorted(CLS_ALL.seed.unique()), hard=True)
check("CNN seed grid complete (15 configs x 3 positions x 10 subjects x 5 seeds)", len(CNN_ALL) == 15 * 3 * 10 * 5, len(CNN_ALL), hard=True)

SK = ["position", "model", "config", "seed", "test_subject", "macro_f1", "macro_f1_workouts"]
SEEDALL = pd.concat([CNN_ALL[SK], CLS_ALL[SK]], ignore_index=True)

cellseed = SEEDALL.groupby(["model", "position", "config", "seed"], as_index=False).macro_f1.mean()
cell = (cellseed.groupby(["model", "position", "config"]).macro_f1
        .agg(n_seeds="count", mean_over_seeds="mean", sd_over_seeds="std", min_over_seeds="min", max_over_seeds="max").reset_index())
cell["range_over_seeds"] = cell.max_over_seeds - cell.min_over_seeds
cell["n_channels"] = cell.config.map(NCH)
sub_mean = SEEDALL.groupby(["model", "position", "config", "test_subject"]).macro_f1
ws = sub_mean.std().groupby(level=[0, 1, 2]).mean().rename("within_subject_seed_sd").reset_index()
bs = sub_mean.mean().groupby(level=[0, 1, 2]).std().rename("between_subject_sd").reset_index()
cell = cell.merge(ws, on=["model", "position", "config"]).merge(bs, on=["model", "position", "config"])
cell["seed_sd_to_between_subject_sd"] = cell.within_subject_seed_sd / cell.between_subject_sd
save_table(cell, "SEED_variability_cells")

sd_ch = (cell.groupby(["model", "n_channels"]).sd_over_seeds.agg(n_cells="count", median_sd="median", mean_sd="mean", max_sd="max").reset_index())
save_table(sd_ch, "SEED_sd_by_model_channels")
display(sd_ch.round(4))
SEED_SD_REF = cell.groupby("model").sd_over_seeds.median()
SEED_SD_MAX = cell.groupby("model").sd_over_seeds.max()
log("Median seed SD of the 10-subject mean macro-F1 by model:", SEED_SD_REF.round(4).to_dict())
for m in MODELS:
    claim(f"seed_sd_median_{m}", f"Median over cells of the SD across seeds of the 10-subject mean macro-F1 ({m}; {int(cell[cell.model == m].n_seeds.max())} seeds)",
          f"{SEED_SD_REF[m]:.4f} (max {SEED_SD_MAX[m]:.4f})", "SEED_variability_cells")

head6 = cell[cell.config == "6-axis"].copy()
head6["mean_pm_sd"] = head6.mean_over_seeds.map("{:.3f}".format) + " +/- " + head6.sd_over_seeds.map("{:.3f}".format)
save_table(head6[["model", "position", "n_seeds", "mean_over_seeds", "sd_over_seeds", "min_over_seeds", "max_over_seeds", "mean_pm_sd"]], "SEED_headline_6axis")
display(head6.pivot(index="model", columns="position", values="mean_pm_sd")[POS])

# noise reference for the single-seed extension
EE["seed_sd_ref"] = EE.model.map(SEED_SD_REF)
EE["effect_gt_3sd"] = EE.mean_diff.abs() > 3 * EE.seed_sd_ref
save_table(EE, "E_all_with_seed_noise_reference")
Ep = EE[EE.metric == PRIMARY_METRIC]
sig_small = Ep[Ep.sig_decrease & ~Ep.effect_gt_3sd]
log(f"Extension: {len(sig_small)} Holm-significant decreases are smaller than 3 x the model's median seed SD (read with caution)")
claim("E_sig_but_small", "Extension tests that are Holm-significant decreases yet smaller than 3x the model's median seed SD", len(sig_small), "E_all_with_seed_noise_reference")

# underfit flags
CNN_ALL["underfit"] = CNN_ALL.final_train_loss > LOSS_FLAG
uf_ch = CNN_ALL.groupby("n_channels").agg(n_units=("underfit", "size"), n_flagged=("underfit", "sum")).reset_index()
uf_ch["share_flagged"] = uf_ch.n_flagged / uf_ch.n_units
uf_cfg = (CNN_ALL.groupby(["position", "config"]).agg(n_units=("underfit", "size"), n_flagged=("underfit", "sum"), max_final_loss=("final_train_loss", "max")).reset_index())
save_table(uf_ch, "SEED_cnn_underfit_by_channels")
save_table(uf_cfg[uf_cfg.n_flagged > 0], "SEED_cnn_underfit_by_config")
ext_cnn = ext[ext.model == "CNN"]
log(f"CNN units with final training loss > {LOSS_FLAG}: {int(CNN_ALL.underfit.sum())} of {len(CNN_ALL)} (primary grid, 5 seeds); "
    f"extended CNN units flagged: {int((ext_cnn.final_train_loss > LOSS_FLAG).sum())} of {len(ext_cnn)}")
display(uf_ch.round(3))
claim("underfit_total", f"CNN primary-grid units (5 seeds) with final train loss > {LOSS_FLAG}", f"{int(CNN_ALL.underfit.sum())} of {len(CNN_ALL)}", "SEED_cnn_underfit_by_channels")
claim("underfit_ext", f"Extended-lattice CNN units with final train loss > {LOSS_FLAG}", f"{int((ext_cnn.final_train_loss > LOSS_FLAG).sum())} of {len(ext_cnn)}", "SEED_cnn_underfit_by_channels")

# primary reduction family re-run per seed and on the seed mean
frames = {}
for s in [42, 43, 44, 45, 46]:
    frames[str(s)] = pd.concat([CNN_ALL[CNN_ALL.seed == s], CLS_ALL[CLS_ALL.seed == s]], ignore_index=True)
frames["mean"] = (pd.concat([CNN_ALL, CLS_ALL], ignore_index=True)
                  .groupby(["position", "model", "config", "test_subject"], as_index=False)[METRICS].mean())
parts = []
for lab, fr in frames.items():
    Vs = build_V(PRIMARY_METRIC, fr)
    t = reduction_table(Vs, P_REDUCED, "primary14", PRIMARY_METRIC, models=[m for m in MODELS if m in set(fr.model)])
    t["seed_label"] = lab
    parts.append(t)
SEED_A = pd.concat(parts, ignore_index=True)
save_table(SEED_A, "SEED_primary_reduction_by_seed")
cnt = (SEED_A.groupby(["model", "seed_label"]).agg(n_tests=("p", "size"), n_sig_decrease=("sig_decrease", "sum"), n_meet_margin=("ni_0p05", "sum")).reset_index())
save_table(cnt, "SEED_holm_counts_by_seed")
display(cnt.pivot(index="model", columns="seed_label", values="n_sig_decrease"))

h04 = locate("tables_seeds", "holm_counts_by_seed.csv", required=False)
if h04 is not None:
    H = pd.read_csv(h04); H["seed"] = H.seed.astype(str)
    mrg = cnt.merge(H, left_on=["model", "seed_label"], right_on=["model", "seed"], how="inner", suffixes=("", "_04c"))
    check("seed-wise Holm counts agree with notebook 04c's table", len(mrg) > 0 and (mrg.n_sig_decrease == mrg.n_sig_decrease_04c).all() and (mrg.n_meet_margin == mrg.n_meet_margin_04c).all(),
          f"{len(mrg)} model x seed rows compared")

mm = SEED_A.groupby(["seed_label", "position", "config"]).agg(n_models=("model", "nunique"), n_meet=("ni_0p05", "sum")).reset_index()
names = (SEED_A[SEED_A.ni_0p05].groupby(["seed_label", "position", "config"]).model.agg(lambda s: ",".join(sorted(s))).rename("models_meeting").reset_index())
MARGIN_SEED = mm.merge(names, on=["seed_label", "position", "config"], how="left")
save_table(MARGIN_SEED[MARGIN_SEED.n_meet > 0], "SEED_margin_map")
all4_seed = MARGIN_SEED[(MARGIN_SEED.n_models == 4) & (MARGIN_SEED.n_meet == 4)]
for lab in ["42", "43", "44", "mean"]:
    g = all4_seed[all4_seed.seed_label == lab]
    txt = "; ".join(f"{r.position}:{r.config}" for r in g.itertuples()) or "(none)"
    log(f"seed {lab}: configurations meeting the -0.05 margin in all four models -> {txt}")
    claim(f"margin_all4_seed_{lab}", f"Primary configurations meeting the -0.05 margin in all four models (seed label {lab})", txt, "SEED_margin_map")

## Block 10 · Per-class effects of the extended configurations (RQ4 extension)

Descriptive mean change in per-class F1 versus 6-axis for all 15 extended configurations. Paired tests only for the declared configuration `mix-AxGz`, per class with at least 6 held-out subjects that contain the class, Holm within position x model across classes. Exploratory.

In [ ]:
pc = U.melt(id_vars=["position", "model", "config", "test_subject"], value_vars=F1COLS, var_name="class", value_name="f1")
pc["class"] = pc["class"].str[3:]
PCD = {}
for key, g in pc.groupby(["position", "model", "class", "config"]):
    PCD[key] = g.set_index("test_subject").f1.reindex(SUBJECTS).values.astype(float)

rows = []
for pos in POS:
    for m in MODELS:
        for c in CLASSES:
            ref_v = PCD[(pos, m, c, "6-axis")]
            for cfg in A1 + A2:
                x = PCD[(pos, m, c, cfg)]
                ok = np.isfinite(x) & np.isfinite(ref_v)
                rows.append(dict(position=pos, model=m, **{"class": c}, config=cfg, kind=KIND[cfg], n_subjects_with_class=int(ok.sum()),
                                 mean_f1_6axis=float(np.nanmean(ref_v)) if np.isfinite(ref_v).any() else np.nan,
                                 mean_drop=float((x - ref_v)[ok].mean()) if ok.any() else np.nan))
PCLS = save_table(pd.DataFrame(rows), "PC_perclass_drop_extended_configs")

rows = []
for pos in POS:
    for m in MODELS:
        for c in CLASSES:
            ref_v = PCD[(pos, m, c, "6-axis")]; x = PCD[(pos, m, c, "mix-AxGz")]
            ok = np.isfinite(x) & np.isfinite(ref_v)
            if ok.sum() >= 6:
                rows.append(dict(position=pos, model=m, **{"class": c}, config="mix-AxGz", n_subjects_with_class=int(ok.sum()), **paired_stats((x - ref_v)[ok])))
PCT = pd.DataFrame(rows)
PCT["fam"] = PCT.position + "|" + PCT.model
PCT = add_holm(PCT, "fam")
save_table(PCT, "PC_mixAxGz_perclass_tests")
print("Per-class: largest mean F1 drops for mix-AxGz (primary tests, Holm within position x model)")
display(PCT.sort_values("mean_diff").head(12)[["position", "model", "class", "mean_diff", "ci_lo", "ci_hi", "n_neg", "p_holm_family"]].round(3))
claim("pc_mixAxGz_sig", "mix-AxGz per-class tests: Holm-significant changes", f"{int((PCT.p_holm_family < ALPHA).sum())} of {len(PCT)}", "PC_mixAxGz_perclass_tests")

## Block 11 · Systems view: data volume, latency, size and Pareto sets

* **Data volume** is an analytic model of raw sample values only (channels x 20 Hz, 16-bit and 32-bit), with no headers, timestamps, transport overhead or compression.
* **Latency, size and complexity** come from notebook 05: one session, one machine, one fold per unit, single window, batch 1, one thread, preparation plus inference. They are **not** edge-device numbers and **not** energy.
* Ratios are always taken against the 6-axis unit of the same position and model.

In [ ]:
LAT = lat.copy()
LAT["size_kb"] = LAT.model_size_bytes / 1024
LAT["complexity_count"] = np.where(LAT.model == "CNN", LAT.n_params, LAT.n_trees_or_sv)
LAT["duty_cycle_pct"] = 100 * LAT.total_median_ms / (HOP_S * 1000)
LAT["prep_share_pct"] = 100 * LAT.prep_median_ms / LAT.total_median_ms
GRID = U.groupby(["position", "model", "config"], as_index=False).agg(macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"))
SYS = LAT.merge(GRID, on=["position", "model", "config"], how="left")
check("every latency unit has a matching macro-F1 cell", SYS.macro_f1_mean.notna().all(), hard=True)
SYS["kind"] = SYS.config.map(KIND)
refcols = ["total_median_ms", "prep_median_ms", "infer_median_ms", "size_kb", "complexity_count"]
ref6 = LAT[LAT.config == "6-axis"][["position", "model"] + refcols].rename(columns={c: c + "_6ax" for c in refcols})
SYS = SYS.merge(ref6, on=["position", "model"], how="left")
for c in refcols:
    SYS[c.replace("_median_ms", "") + "_ratio"] = SYS[c] / SYS[c + "_6ax"]
SYS["data_rate_hz"] = SYS.n_channels * FS
save_table(SYS, "SYS_latency_size_f1_merged")

# one-session QC from the latency run
qcp = locate("checks", "latency_qc.json", required=False)
if qcp is not None:
    qc = json.load(open(qcp))
    log("latency QC:", {k: qc[k] for k in ["n_units", "pass_range_pct_median", "pass_range_pct_max", "share_units_pass_range_over_10pct"] if k in qc})
log("latency machine(s):", sorted(set(LAT.cpu_model.astype(str))), "| sessions:", sorted(set(LAT.session_id.astype(str))) if "session_id" in LAT else "n/a")
check("latency measured in a single session on a single machine", LAT.machine_id.nunique() == 1 if "machine_id" in LAT else True, LAT.machine_id.unique().tolist() if "machine_id" in LAT else "")

# analytic data volume
DV = pd.DataFrame([dict(config=c, group=GROUP[c], n_channels=NCH[c], channels="".join(sorted(cfg_axes(c), key=IMU.index)),
                        samples_per_s=NCH[c] * FS, window_values=NCH[c] * 80, reduction_vs_6axis_pct=100 * (1 - NCH[c] / 6),
                        bytes_per_s_16bit=NCH[c] * FS * 2, bytes_per_s_32bit=NCH[c] * FS * 4,
                        MB_per_hour_16bit=NCH[c] * FS * 2 * 3600 / 1e6, MB_per_hour_32bit=NCH[c] * FS * 4 * 3600 / 1e6) for c in ALL30])
save_table(DV, "SYS_data_volume_analytic")
l4p = locate("tables", "TableL4_data_volume.csv", required=False)
if l4p is not None:
    L4 = pd.read_csv(l4p).merge(DV[["config", "samples_per_s", "MB_per_hour_16bit"]], on="config", suffixes=("", "_mine"))
    check("analytic data volume agrees with notebook 05's TableL4", (L4.samples_per_s == L4.samples_per_s_mine).all() and np.allclose(L4.MB_per_hour_16bit, L4.MB_per_hour_16bit_mine), f"{len(L4)} configs")

# cost by channel count and trend per model
SYS["prep_share_pct"] = 100 * SYS.prep_median_ms / SYS.total_median_ms
LAT_K = (SYS.groupby(["model", "n_channels"]).agg(n_units=("config", "size"), total_ms_median=("total_median_ms", "median"),
                                                  total_ratio_median=("total_ratio", "median"),
                                                  total_ratio_q25=("total_ratio", lambda s: s.quantile(0.25)),
                                                  total_ratio_q75=("total_ratio", lambda s: s.quantile(0.75)),
                                                  prep_share_pct_median=("prep_share_pct", "median"),
                                                  size_kb_median=("size_kb", "median"), size_ratio_median=("size_kb_ratio", "median"),
                                                  complexity_ratio_median=("complexity_count_ratio", "median"),
                                                  duty_cycle_pct_median=("duty_cycle_pct", "median")).reset_index())
save_table(LAT_K, "SYS_cost_by_channel_count")
display(LAT_K.round(3))

rows = []
for m in MODELS:
    g = SYS[SYS.model == m]
    rho, p = spearmanr(g.n_channels, g.total_ratio)
    slope = np.polyfit(g.n_channels.astype(float), g.total_ratio.values, 1)[0]
    r1 = LAT_K[(LAT_K.model == m) & (LAT_K.n_channels == 1)].total_ratio_median.iloc[0]
    rs = LAT_K[(LAT_K.model == m) & (LAT_K.n_channels == 1)].size_ratio_median.iloc[0]
    rows.append(dict(model=m, n_units=len(g), spearman_latency_ratio_vs_channels=float(rho), p=float(p),
                     latency_ratio_change_per_channel=float(slope), median_latency_ratio_at_1_channel=float(r1), median_size_ratio_at_1_channel=float(rs)))
    claim(f"sys_latency_{m}", f"{m}: Spearman rho between channel count and latency ratio vs 6-axis (90 units); median latency ratio at 1 channel; median size ratio at 1 channel",
          f"rho {rho:+.2f}; {r1:.2f}; {rs:.2f}", "SYS_trend_by_model")
TREND = save_table(pd.DataFrame(rows), "SYS_trend_by_model")
display(TREND.round(3))

# Pareto sets
def pareto_mask(cost, value):
    cost = np.asarray(cost, float); value = np.asarray(value, float)
    keep = np.ones(len(cost), bool)
    for i in range(len(cost)):
        dom = (cost <= cost[i]) & (value >= value[i]) & ((cost < cost[i]) | (value > value[i]))
        if dom.any():
            keep[i] = False
    return keep

parts = []
for (pos, m), g in SYS.groupby(["position", "model"]):
    g = g.copy()
    g["pareto_data_rate"] = pareto_mask(g.data_rate_hz, g.macro_f1_mean)
    g["pareto_latency"] = pareto_mask(g.total_median_ms, g.macro_f1_mean)
    parts.append(g)
PARETO = pd.concat(parts, ignore_index=True)
save_table(PARETO[["position", "model", "config", "group", "kind", "n_channels", "data_rate_hz", "total_median_ms", "size_kb", "macro_f1_mean", "pareto_data_rate", "pareto_latency"]], "SYS_pareto_sets")
rec = PARETO[PARETO.pareto_data_rate].groupby(["kind", "config"]).size().rename("n_cells_on_front_of_12").reset_index().sort_values("n_cells_on_front_of_12", ascending=False)
save_table(rec, "SYS_pareto_recurrence")
print("Configurations that sit on the macro-F1 vs data-rate Pareto front, counted over the 12 position x model cells:")
display(rec.head(12))
claim("pareto_front_mixed_share", "Share of Pareto-front points (macro-F1 vs data rate, 12 cells) that are mixed pairs",
      f"{(PARETO[PARETO.pareto_data_rate].kind == 'mixed-pair').mean():.3f}", "SYS_pareto_sets")

# smallest channel count that retains a given share of 6-axis macro-F1 (best-of is optimistic; selection-free share shown too)
rows = []
for pos in POS:
    for m in MODELS:
        m6 = vec(V1, pos, m, "6-axis").mean()
        for thr in (0.95, 0.90, 0.85):
            for k in (1, 2, 3, 5):
                cs = [c for c in ALL30 if NCH[c] == k]
                ret = pd.Series({c: vec(V1, pos, m, c).mean() / m6 for c in cs})
                rows.append(dict(position=pos, model=m, threshold=thr, n_channels=k, n_configs=len(cs), n_meeting=int((ret >= thr).sum()),
                                 share_of_configs_meeting=float((ret >= thr).mean()), best_cfg=ret.idxmax(), best_retained=float(ret.max())))
RET = save_table(pd.DataFrame(rows), "SYS_retention_threshold_by_channel_count")
minimal = (RET[RET.n_meeting > 0].groupby(["position", "model", "threshold"]).n_channels.min().rename("fewest_channels_with_a_config_meeting").reset_index())
save_table(minimal, "SYS_fewest_channels_for_retention")
display(minimal.pivot_table(index=["position", "model"], columns="threshold", values="fewest_channels_with_a_config_meeting"))

## Block 12 · Extended master table, claims re-check, claims-to-sources manifest

The extended master table has one row per position x model x configuration (360 rows). Statistics come from the family each configuration belongs to (primary 14, E1 or E2). The claims table re-evaluates, from the data, each statement that the handoff flagged as result-dependent (Part 6.4).

In [ ]:
G = U.groupby(["position", "model", "config"], as_index=False).agg(
    macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"), macro_f1_median=("macro_f1", "median"),
    macro_f1_min=("macro_f1", "min"), macro_f1_max=("macro_f1", "max"), accuracy_mean=("accuracy", "mean"),
    weighted_f1_mean=("weighted_f1", "mean"), macro_f1_workouts_mean=("macro_f1_workouts", "mean"))
ST = pd.concat([Ap, EE[EE.metric == PRIMARY_METRIC]], ignore_index=True)[
    ["position", "model", "config", "family", "mean_diff", "ci_lo", "ci_hi", "dz", "rbc", "n_pos", "n_neg", "p", "p_holm_family", "ni_0p05", "sig_decrease"]]
MASTER = G.merge(ST, on=["position", "model", "config"], how="left")
MASTER["family"] = MASTER.family.fillna("reference (6-axis)")
m6 = MASTER[MASTER.config == "6-axis"].set_index(["position", "model"]).macro_f1_mean
MASTER["retained_pct"] = [100 * v / m6[(p, m)] for v, p, m in zip(MASTER.macro_f1_mean, MASTER.position, MASTER.model)]
MASTER["group"] = MASTER.config.map(GROUP)
MASTER["kind"] = MASTER.config.map(KIND)
MASTER["n_channels"] = MASTER.config.map(NCH)
MASTER["data_rate_hz"] = MASTER.n_channels * FS
MASTER["axis_channels"] = MASTER.config.map(lambda c: "+".join(sorted(cfg_axes(c), key=IMU.index)))
MASTER = MASTER.merge(cell[["model", "position", "config", "n_seeds", "mean_over_seeds", "sd_over_seeds"]], on=["model", "position", "config"], how="left")
MASTER["n_seeds"] = MASTER.n_seeds.fillna(1).astype(int)
MASTER = MASTER.merge(SYS[["position", "model", "config", "total_median_ms", "prep_median_ms", "infer_median_ms", "size_kb", "complexity_count", "duty_cycle_pct"]],
                      on=["position", "model", "config"], how="left")
lead = ["position", "model", "config", "group", "kind", "family", "axis_channels", "n_channels", "data_rate_hz"]
MASTER = MASTER[lead + [c for c in MASTER.columns if c not in lead]]
check("extended master table has 360 rows", len(MASTER) == 360, len(MASTER), hard=True)
save_table(MASTER, "master_results_extended")

DICT = [("position", "wrist, leg (calf) or pocket"), ("model", "RF, SVM, XGB or CNN"), ("config", "axis configuration name"),
        ("group", "P primary grid, A1 drop-one, A2 mixed pairs"), ("kind", "structural type of the configuration"),
        ("family", "statistical family: primary14, E1_drop_one or E2_mixed_pairs (Holm within position x model)"),
        ("axis_channels", "device-frame channels retained"), ("n_channels", "number of channels"), ("data_rate_hz", "channels x 20 Hz"),
        ("macro_f1_mean", "mean over 10 held-out subjects, seed 42, 12 labels incl. Null"), ("macro_f1_sd", "SD over the 10 subjects"),
        ("macro_f1_median", "median over subjects"), ("macro_f1_min", "minimum over subjects"), ("macro_f1_max", "maximum over subjects"),
        ("accuracy_mean", "mean accuracy"), ("weighted_f1_mean", "mean weighted F1"), ("macro_f1_workouts_mean", "mean macro-F1 over the 11 workouts (Null excluded)"),
        ("mean_diff", "mean paired difference vs the 6-axis result of the same subjects (negative = loss)"),
        ("ci_lo", "lower bound of the percentile bootstrap 95 % CI over subjects"), ("ci_hi", "upper bound"),
        ("dz", "Cohen dz of the paired differences"), ("rbc", "rank-biserial correlation"), ("n_pos", "subjects with a higher score than 6-axis"),
        ("n_neg", "subjects with a lower score than 6-axis"), ("p", "two-sided Wilcoxon signed-rank p (unadjusted)"),
        ("p_holm_family", "Holm-adjusted p within the position x model family"), ("ni_0p05", "True if the CI lower bound exceeds -0.05 (descriptive margin)"),
        ("sig_decrease", "Holm-significant decrease vs 6-axis"), ("retained_pct", "100 x mean macro-F1 / 6-axis mean macro-F1 of the same position and model"),
        ("n_seeds", "number of training seeds behind this cell (1 for the extended configurations)"),
        ("mean_over_seeds", "mean over seeds of the 10-subject mean macro-F1 (primary configurations only)"),
        ("sd_over_seeds", "SD over seeds of the 10-subject mean macro-F1 (primary configurations only)"),
        ("total_median_ms", "median single-window latency, preparation plus inference, one machine, one fold (notebook 05)"),
        ("prep_median_ms", "median preparation time (slice, features or z-score)"), ("infer_median_ms", "median inference time"),
        ("size_kb", "serialised model size in kB (notebook 05)"), ("complexity_count", "CNN parameters, or trees / support vectors for classical models"),
        ("duty_cycle_pct", "latency as a share of the 2 s window hop on one core of that CPU; not an energy figure")]
save_table(pd.DataFrame(DICT, columns=["column", "description"]), "master_results_extended_dictionary")

# ---------------- claims re-check (Part 6.4 of the handoff) ----------------
ALLRED = pd.concat([Ap, Ep], ignore_index=True)
tab = ALLRED.groupby(["position", "config"]).agg(n_models=("model", "nunique"), n_meet=("ni_0p05", "sum")).reset_index()
all4 = tab[(tab.n_models == 4) & (tab.n_meet == 4)]
all4_list = [f"{r.position}:{r.config}" for r in all4.itertuples()]
log("Configurations meeting the -0.05 margin in all four models (seed 42, all 30 configurations):", all4_list or "(none)")
claim("margin_all4_seed42_all30", "Configurations (of 29 reduced, over 3 positions) meeting the -0.05 margin in all four models, seed 42, extended lattice included",
      "; ".join(all4_list) or "(none)", "A_primary_reduction_vs_6axis + E1_drop_one_vs_6axis + E2_mixed_pairs_vs_6axis")
only_leg = (all4_list == ["leg:gyro-3"])
claim("claim_only_leg_gyro3", "Is calf gyro-3 still the only configuration meeting the margin in all four models (seed 42, extended lattice)?", only_leg, "A_primary_reduction_vs_6axis + E1_drop_one_vs_6axis + E2_mixed_pairs_vs_6axis")
claim("claim_no_mixed_configs", "Statement 'no configuration mixes accelerometer and gyroscope axes' is superseded: mixed pairs evaluated", len(MIXED), "E2_mixed_pairs_vs_6axis")

sg = Ap[(Ap.config == "single-Gz") & (Ap.position == "leg")]
claim("leg_single_gz", "Calf single-Gz: loss vs 6-axis and retained % per model",
      "; ".join(f"{r.model} {r.mean_diff:+.3f} ({r.retained_pct:.1f}%)" for r in sg.itertuples()), "A_primary_reduction_vs_6axis")
best1 = BMW[BMW.n_channels == 1]
claim("best_single_by_cell", "Best single axis per position (modal over models)",
      "; ".join(f"{p}: {best1[best1.position == p].best_cfg.value_counts().index[0]}" for p in POS), "BMW_best_median_worst_by_channel_count")

rows = []
for pos in POS:
    for k in (5, 3, 2, 1):
        b = BMW[(BMW.position == pos) & (BMW.n_channels == k)]
        modal = b.best_cfg.value_counts().index[0]
        sub = ALLRED[(ALLRED.position == pos) & (ALLRED.config == modal)]
        rows.append(dict(position=pos, n_channels=k, modal_best_config=modal, n_models_agreeing=int((b.best_cfg == modal).sum()),
                         mean_retained_pct_of_modal=float(sub.retained_pct.mean()), median_retained_pct_all_configs_of_size=float(b.median_retained_pct.mean()),
                         worst_retained_pct_all_configs_of_size=float(b.worst_retained_pct.mean()), lattice_complete=bool(b.lattice_complete.iloc[0])))
GUIDE = save_table(pd.DataFrame(rows), "GUIDE_practical_by_position_and_channel_count")
display(GUIDE.round(1))
for r in GUIDE.itertuples():
    claim(f"guide_{r.position}_{r.n_channels}", f"{r.position}, {r.n_channels} channels: modal best configuration (models agreeing), its mean retained %",
          f"{r.modal_best_config} ({r.n_models_agreeing}/4), {r.mean_retained_pct_of_modal:.1f}%", "GUIDE_practical_by_position_and_channel_count")

# primary results by position, for the digest
for pos in POS:
    s6 = MASTER[(MASTER.position == pos) & (MASTER.config == "6-axis")].set_index("model").macro_f1_mean
    claim(f"six_axis_{pos}", f"6-axis macro-F1 at {pos} (RF, SVM, XGB, CNN; seed 42)", ", ".join(f"{m} {s6[m]:.3f}" for m in MODELS), "master_results_extended")

CL = pd.DataFrame(CLAIMS)
CL["source_table_exists"] = CL.source_table.map(lambda s: all(t.strip() in SAVED_TABLES for t in s.split("+")))
save_table(CL, "claims_to_sources_manifest")
display(CL[["claim_id", "value", "source_table"]].head(40))

In [ ]:
# ---------------------------- findings digest (every number computed above; nothing typed in) ----------------------------
L = []
L.append("# Findings digest (auto-generated by notebook 06)\n")
L.append(f"Run settings: N_BOOT = {N_BOOT}, QUICK = {QUICK}. Seed-42 grid for all 30 configurations; extension has one seed.\n")
n_fail = sum(1 for c in CHECKS if not c["passed"])
L.append(f"## Integrity\n- {len(CHECKS)} checks, {n_fail} not passed (hard checks stop the notebook; warnings are listed in checks/integrity_checks.json).")
L.append(f"- Primary grid: {n_sig_primary} of 168 reduced-vs-6-axis comparisons are Holm-significant decreases (recomputed).\n")
L.append("## Extended lattice vs 6-axis (Holm per position x model)")
for fam, g in Ep.groupby("family"):
    L.append(f"- {fam}: {int(g.sig_decrease.sum())} of {len(g)} Holm-significant decreases; {int(g.ni_0p05.sum())} tests with CI lower bound above -0.05.")
for pos in POS:
    L.append(f"- Most damaging axis to remove at {pos}: {worst_axis[pos]} ({agree[pos]} of 4 models agree).")
L.append("")
L.append("## Selection-bias control")
L.append(f"- Best-of optimism (naive best minus leave-one-subject-out selected): median {SEL.optimism.median():.4f}, max {SEL.optimism.max():.4f} macro-F1 over all cells.")
for r in GUIDE.itertuples():
    L.append(f"- {r.position}, {r.n_channels} channels: modal best {r.modal_best_config} ({r.n_models_agreeing}/4 models), retained {r.mean_retained_pct_of_modal:.1f}% ; all configurations of that size: median {r.median_retained_pct_all_configs_of_size:.1f}%, worst {r.worst_retained_pct_all_configs_of_size:.1f}%{'' if r.lattice_complete else ' (lattice incomplete at this size)'}.")
L.append("")
L.append("## Mixed versus pure pairs")
L.append(f"- Best 2-channel configuration is a mixed pair in {n_mixed_best} of 12 cells.")
L.append(f"- Selection-free (mixed minus all pure pairs): positive in {int((sf.mean_diff > 0).sum())} of 12 cells, Holm-significant in {int((sf.p_holm_family < ALPHA).sum())}.")
L.append(f"- Declared exploratory mix-AxGz minus best pure pair: mean difference from {ex.mean_diff.min():+.3f} to {ex.mean_diff.max():+.3f}; CI lower bound above 0 in {int((ex.ci_lo > 0).sum())} of 12 cells (comparator inflated by same-fold selection).\n")
L.append("## Claims that depended on the old lattice")
L.append(f"- Configurations meeting the -0.05 margin in all four models (seed 42, 30 configurations): {', '.join(all4_list) or 'none'}. Still only calf gyro-3: {only_leg}.")
for lab in ["42", "43", "44", "mean"]:
    g = all4_seed[all4_seed.seed_label == lab]
    L.append(f"  - primary configurations, seed label {lab}: {', '.join(f'{r.position}:{r.config}' for r in g.itertuples()) or 'none'}")
L.append("")
L.append("## Seeds")
for m in MODELS:
    L.append(f"- {m}: median seed SD {SEED_SD_REF[m]:.4f}, max {SEED_SD_MAX[m]:.4f} (n seeds = {int(cell[cell.model == m].n_seeds.max())}).")
L.append(f"- CNN units with final train loss > {LOSS_FLAG}: {int(CNN_ALL.underfit.sum())} of {len(CNN_ALL)} (primary grid); extended CNN: {int((ext_cnn.final_train_loss > LOSS_FLAG).sum())} of {len(ext_cnn)}.\n")
L.append("## Model-ranking agreement (k = 2, 15 configurations)")
for r in RANK[RANK.config_set == "k=2 (15 pairs)"].itertuples():
    L.append(f"- {r.position}: Kendall W {r.kendall_w:.3f} (95% CI {r.w_ci_lo:.3f} to {r.w_ci_hi:.3f}); mean pairwise Spearman {r.mean_pairwise_spearman:.3f}.")
L.append("")
L.append("## Systems (single machine, single fold per unit; not edge-device latency, not energy)")
for r in TREND.itertuples():
    L.append(f"- {r.model}: Spearman(latency ratio, channels) {r.spearman_latency_ratio_vs_channels:+.2f}; median latency ratio at 1 channel {r.median_latency_ratio_at_1_channel:.2f}; median size ratio at 1 channel {r.median_size_ratio_at_1_channel:.2f}.")
L.append("- Data volume scales exactly with channel count (analytic: channels x 20 Hz).")
DIGEST = "\n".join(L)
(OUT / "findings_digest.md").write_text(DIGEST)
display(Markdown(DIGEST))

## Block 13 · Figures

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
COL = {"RF": "#1b9e77", "SVM": "#d95f02", "XGB": "#7570b3", "CNN": "#e7298a"}
KCOL = {"full": "black", "drop-one": "#1f78b4", "pure-triple": "#33a02c", "acc-pair": "#b2df8a", "gyro-pair": "#a6cee3",
        "mixed-pair": "#e31a1c", "single-acc": "#fdbf6f", "single-gyro": "#cab2d6"}

def heat(ax, M, rl, cl, title="", vmin=None, vmax=None, fmt="{:.2f}", cmap="viridis", fs=7):
    im = ax.imshow(M, aspect="auto", vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xticks(range(len(cl))); ax.set_xticklabels(cl, rotation=60, ha="right", fontsize=fs + 1)
    ax.set_yticks(range(len(rl))); ax.set_yticklabels(rl, fontsize=fs + 1)
    ax.set_title(title, fontsize=9)
    lo = np.nanmin(M) if vmin is None else vmin
    hi = np.nanmax(M) if vmax is None else vmax
    thr = (lo + hi) / 2
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isfinite(v):
                ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=fs, color="white" if (v < thr and cmap == "viridis") else "black")
    return im

# FX1: macro-F1 of all 30 configurations
cols = [(p, m) for p in POS for m in MODELS]
Mh = np.array([[vec(V1, p, m, c).mean() for (p, m) in cols] for c in ALL30])
fig, ax = plt.subplots(figsize=(13, 11))
im = heat(ax, Mh, ALL30, [f"{p}\n{m}" for p, m in cols], "Mean macro-F1 over 10 held-out subjects, all 30 configurations (seed 42)", vmin=0.2, vmax=0.95)
fig.colorbar(im, ax=ax, shrink=0.8)
savefig(fig, "FX1_macroF1_heatmap_30_configs")

# FX2: E1 and E2 vs 6-axis
cf = A1 + A2
fig, axes = plt.subplots(1, 3, figsize=(19, 5), sharey=True)
for ax, pos in zip(axes, POS):
    for k, m in enumerate(MODELS):
        s = Ep[(Ep.position == pos) & (Ep.model == m)].set_index("config").loc[cf]
        x = np.arange(len(cf)) + (k - 1.5) * 0.15
        ax.errorbar(x, s.mean_diff.values, yerr=[(s.mean_diff - s.ci_lo).values, (s.ci_hi - s.mean_diff).values], fmt="none", ecolor=COL[m], alpha=0.55, lw=1)
        sig = s.sig_family.values.astype(bool)
        ax.scatter(x[sig], s.mean_diff.values[sig], color=COL[m], s=26, zorder=3)
        ax.scatter(x[~sig], s.mean_diff.values[~sig], facecolors="white", edgecolors=COL[m], s=26, zorder=3)
    ax.axhline(0, color="k", lw=0.8); ax.axhline(-MARGIN, color="grey", ls=":", lw=0.8)
    ax.axvline(len(A1) - 0.5, color="grey", lw=0.6)
    ax.set_xticks(range(len(cf))); ax.set_xticklabels(cf, rotation=60, ha="right"); ax.set_title(pos)
axes[0].set_ylabel("Mean paired difference in macro-F1 vs 6-axis")
axes[0].legend(handles=[Line2D([0], [0], marker="o", color=COL[m], lw=0, label=m) for m in MODELS], frameon=False, loc="lower left")
fig.suptitle("E1 (drop-one) and E2 (mixed pairs) vs 6-axis: filled = Holm-significant within position x model; bars = bootstrap 95% CI; dotted = -0.05", y=1.02)
savefig(fig, "FX2_E1_E2_vs_6axis")

# FX3: best / median / worst per channel count
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)
for ax, pos in zip(axes, POS):
    for m in MODELS:
        s = BMW[(BMW.position == pos) & (BMW.model == m)].sort_values("n_channels")
        ax.plot(s.n_channels, s["best"], color=COL[m], lw=1.6, marker="o", ms=4)
        ax.plot(s.n_channels, s["median"], color=COL[m], lw=1.0, ls="--")
        ax.plot(s.n_channels, s["worst"], color=COL[m], lw=1.0, ls=":")
    ax.set_xticks([1, 2, 3, 5, 6]); ax.set_xlabel("channels"); ax.set_title(pos)
axes[0].set_ylabel("Mean macro-F1 over subjects")
axes[0].legend(handles=[Line2D([0], [0], color=COL[m], lw=2, label=m) for m in MODELS] +
               [Line2D([0], [0], color="k", lw=1.6, label="best"), Line2D([0], [0], color="k", lw=1, ls="--", label="median"), Line2D([0], [0], color="k", lw=1, ls=":", label="worst")],
               frameon=False, fontsize=7, loc="lower right")
fig.suptitle("Best, median and worst configuration per channel count (k=3 has only the two pure triples; k=4 not evaluated)", y=1.02)
savefig(fig, "FX3_best_median_worst_by_channel_count")

# FX4: Pareto grid
fig, axes = plt.subplots(3, 4, figsize=(17, 11), sharex=True)
jit = np.random.default_rng(1)
for i, pos in enumerate(POS):
    for j, m in enumerate(MODELS):
        ax = axes[i, j]
        g = PARETO[(PARETO.position == pos) & (PARETO.model == m)]
        for kd, gg in g.groupby("kind"):
            ax.scatter(gg.data_rate_hz + jit.uniform(-2.5, 2.5, len(gg)), gg.macro_f1_mean, color=KCOL.get(kd, "grey"), s=24, alpha=0.85)
        p = g[g.pareto_data_rate].sort_values("data_rate_hz")
        ax.plot(p.data_rate_hz, p.macro_f1_mean, color="k", lw=0.8)
        ax.scatter(p.data_rate_hz, p.macro_f1_mean, facecolors="none", edgecolors="k", s=75)
        ax.set_title(f"{pos} | {m}", fontsize=9)
        if i == 2: ax.set_xlabel("data rate (Hz) = channels x 20")
        if j == 0: ax.set_ylabel("macro-F1")
fig.legend(handles=[Line2D([0], [0], marker="o", color=KCOL[k], lw=0, label=k) for k in KCOL], loc="lower center", ncol=8, frameon=False)
fig.suptitle("Macro-F1 vs data rate for all 30 configurations; circled = Pareto-optimal (seed 42)", y=0.995)
savefig(fig, "FX4_pareto_f1_vs_data_rate")

# FX5: pairs by type
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=False)
cats = [("accelerometer pairs", ACC_PAIRS), ("gyroscope pairs", GYR_PAIRS), ("mixed pairs", MIXED)]
for ax, pos in zip(axes, POS):
    for k, m in enumerate(MODELS):
        for xi, (nm, cs) in enumerate(cats):
            vals = [vec(V1, pos, m, c).mean() for c in cs]
            ax.scatter(np.full(len(vals), xi) + (k - 1.5) * 0.12, vals, color=COL[m], s=18, alpha=0.8)
            if "mix-AxGz" in cs:
                ax.scatter([xi + (k - 1.5) * 0.12], [vec(V1, pos, m, "mix-AxGz").mean()], facecolors="none", edgecolors="k", s=70)
    ax.set_xticks(range(3)); ax.set_xticklabels([c[0] for c in cats]); ax.set_title(pos)
axes[0].set_ylabel("Mean macro-F1 of each pair"); axes[0].legend(handles=[Line2D([0], [0], marker="o", color=COL[m], lw=0, label=m) for m in MODELS], frameon=False)
fig.suptitle("Two-channel configurations by type (circled = mix-AxGz)", y=1.02)
savefig(fig, "FX5_pairs_by_type")

# FX6: model-ranking agreement
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for r_, sname in enumerate(["k=2 (15 pairs)", "all 30 configs"]):
    for c_, pos in enumerate(POS):
        W = RANK[(RANK.position == pos) & (RANK.config_set == sname)].kendall_w.iloc[0]
        heat(axes[r_, c_], RHO[(sname, pos)], MODELS, MODELS, f"{pos} | {sname} | W = {W:.2f}", vmin=-1, vmax=1, cmap="RdBu_r", fs=8)
fig.suptitle("Spearman correlation between models of the configuration ranking", y=1.01)
savefig(fig, "FX6_model_ranking_agreement")

# FX7: seed SD by channel count
fig, ax = plt.subplots(figsize=(8, 4.2))
for k, m in enumerate(MODELS):
    g = cell[cell.model == m]
    ax.scatter(g.n_channels + (k - 1.5) * 0.12, g.sd_over_seeds, color=COL[m], s=14, alpha=0.6)
    med = g.groupby("n_channels").sd_over_seeds.median()
    ax.scatter(med.index + (k - 1.5) * 0.12, med.values, color=COL[m], marker="_", s=260, lw=2)
ax.set_xlabel("channels"); ax.set_ylabel("SD across seeds of the 10-subject mean macro-F1"); ax.set_xticks([1, 2, 3, 6])
ax.legend(handles=[Line2D([0], [0], marker="o", color=COL[m], lw=0, label=m) for m in MODELS], frameon=False)
ax.set_title("Seed variability of primary cells (dash = median)")
savefig(fig, "FX7_seed_sd_by_channels")

# FX8: systems
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
ks = [1, 2, 3, 5, 6]
axes[0].plot(ks, [2 * FS * k for k in ks], marker="o", color="k")
axes[0].set_ylabel("raw payload, bytes/s (16-bit samples)"); axes[0].set_title("Analytic data volume")
for m in MODELS:
    s = LAT_K[LAT_K.model == m].sort_values("n_channels")
    axes[1].errorbar(s.n_channels, s.total_ratio_median, yerr=[(s.total_ratio_median - s.total_ratio_q25).values, (s.total_ratio_q75 - s.total_ratio_median).values], color=COL[m], marker="o", ms=4, capsize=2, label=m)
    axes[2].plot(s.n_channels, s.size_ratio_median, color=COL[m], marker="o", ms=4, label=m)
axes[1].set_ylabel("latency ratio vs 6-axis (median, IQR over units)"); axes[1].set_title("Single-window latency (one machine, not edge)")
axes[2].set_ylabel("model-size ratio vs 6-axis"); axes[2].set_title("Serialised model size")
for ax in axes:
    ax.set_xticks(ks); ax.set_xlabel("channels")
axes[1].axhline(1, color="grey", lw=0.6); axes[2].axhline(1, color="grey", lw=0.6)
axes[1].legend(frameon=False)
savefig(fig, "FX8_systems_vs_channels")

# FX9: complementarity
fig, ax = plt.subplots(figsize=(9, 4.2))
for k, m in enumerate(MODELS):
    s = COMP[COMP.model == m].set_index("position").loc[POS]
    x = np.arange(3) + (k - 1.5) * 0.18
    ax.bar(x, s.mean_diff.values, width=0.17, color=COL[m], alpha=0.85, label=m)
    ax.errorbar(x, s.mean_diff.values, yerr=[(s.mean_diff - s.ci_lo).values, (s.ci_hi - s.mean_diff).values], fmt="none", ecolor="k", lw=0.9)
ax.axhline(0, color="k", lw=0.8); ax.set_xticks(range(3)); ax.set_xticklabels(POS)
ax.set_ylabel("mixed minus pure pairs: gain over the better single axis"); ax.legend(frameon=False)
ax.set_title("Complementarity of mixed pairs relative to pure pairs (bars = bootstrap 95% CI)")
savefig(fig, "FX9_complementarity_mixed_vs_pure")

# FX10: per-class drop of mix-AxGz
sub = PCLS[PCLS.config == "mix-AxGz"]
cols = [(p, m) for p in POS for m in MODELS]
Mc = np.array([[sub[(sub.position == p) & (sub.model == m) & (sub["class"] == c)].mean_drop.iloc[0] for (p, m) in cols] for c in CLASSES])
fig, ax = plt.subplots(figsize=(11, 5.5))
im = heat(ax, Mc, CLASSES, [f"{p}\n{m}" for p, m in cols], "Mean per-class F1 change of mix-AxGz vs 6-axis", vmin=-0.4, vmax=0.4, cmap="RdBu", fmt="{:+.2f}", fs=6)
fig.colorbar(im, ax=ax, shrink=0.8)
savefig(fig, "FX10_perclass_mixAxGz")

# FX11: optimism of best-of
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, pos in zip(axes, POS):
    for k, m in enumerate(MODELS):
        s = SEL[(SEL.position == pos) & (SEL.model == m)].sort_values("n_channels")
        ax.bar(np.arange(len(s)) + (k - 1.5) * 0.18, s.optimism.values, width=0.17, color=COL[m], label=m)
    ax.set_xticks(range(4)); ax.set_xticklabels(["k=1", "k=2", "k=3", "k=5"]); ax.set_title(pos); ax.axhline(0, color="k", lw=0.6)
axes[0].set_ylabel("naive best minus LOSO-selected macro-F1"); axes[0].legend(frameon=False)
fig.suptitle("Approximate optimism of best-of-size numbers (leave-one-subject-out selection on stored scores)", y=1.02)
savefig(fig, "FX11_selection_optimism")

In [ ]:
# ---------------------------- manifest, checks, archive ----------------------------
n_warn = sum(1 for c in CHECKS if not c["passed"])
manifest = {
    "notebook": "06_analysis_extended.ipynb", "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "parameters": {"SEED": SEED, "N_BOOT": N_BOOT, "N_BOOT_RANK": N_BOOT_RANK, "QUICK": QUICK, "ALPHA": ALPHA, "DELTAS": DELTAS, "MARGIN": MARGIN,
                   "PRIMARY_METRIC": PRIMARY_METRIC, "SENS_METRIC": SENS_METRIC, "FS": FS, "LOSS_FLAG": LOSS_FLAG},
    "inputs": {k: {"path": v, "sha256": sha256_file(v)} for k, v in INPUT_FILES.items() if v.endswith(".csv") and k.endswith("results.csv")},
    "all_input_files_used": INPUT_FILES,
    "lattice": {"configs": ALL30, "evaluated_by_channel_count": {str(k): int(sum(1 for c in ALL30 if NCH[c] == k)) for k in (1, 2, 3, 5, 6)},
                "not_run": ["A3 mixed triples", "A4 four-channel subsets", "second dataset", "nested LOSO selection (WP-C2)"]},
    "seeds": {"CNN": sorted(CNN_ALL.seed.unique().tolist()), "classical": sorted(CLS_ALL.seed.unique().tolist()), "extended_configs": [42]},
    "n_checks": len(CHECKS), "n_checks_not_passed": n_warn,
    "n_claims": len(CLAIMS), "n_tables": len(SAVED_TABLES),
    "packages": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__, "matplotlib": matplotlib.__version__},
    "notes": ["Wilcoxon minimum exact p at n=10 is 2/1024", "Axes are device-frame channels", "No energy claim; latency is single-machine, single-fold, batch 1, one thread",
              "Leave-one-subject-out selection on stored scores is approximate, not nested LOSO",
              "Extended configurations have one seed; seed SD of primary cells is used as noise reference"],
}
json.dump(manifest, open(OUT / "checks" / "analysis_manifest.json", "w"), indent=2, default=str)
json.dump(CHECKS, open(OUT / "checks" / "integrity_checks.json", "w"), indent=2)
(OUT / "analysis_log.txt").write_text("\n".join(LOG))
log(f"{len(CHECKS)} checks, {n_warn} not passed, {len(CLAIMS)} claims registered, {len(SAVED_TABLES)} tables")
bad = [c for c in CHECKS if not c["passed"]]
if bad:
    print("Checks not passed (non-blocking):")
    for c in bad:
        print(" -", c["check"], "|", c["detail"])
zpath = shutil.make_archive(str(OUT.parent / "recgym_06_analysis_extended"), "zip", OUT)
log("Archive written:", zpath)
try:
    from IPython.display import FileLink
    display(FileLink(zpath))
except Exception:
    pass